# How far are KV-cache eviction policies from the offline optimum?
**Design & Analysis of Algorithms project.** Server benchmark that runs every algorithm through every test. Default model: Qwen2.5-0.5B.

**Problem.** While generating text, an LLM keeps the keys/values (the *KV cache*) of all previous tokens. With limited memory only `k` of them fit, so at every step an **eviction policy** decides which token to drop. Eviction is **permanent**. This is an **online problem**, like page replacement. We compare practical online policies with the **exact offline optimum**, which we compute by a reduction to **min-cost flow**.

**Algorithms (all of them run through all tests):**
- **Base:** Random, SlidingWindow, StreamingLLM, HeavyHitter-only, H2O, Hybrid, Belady-Greedy (offline), Optimal (offline). H2O, Hybrid and HeavyHitter-only rank by *average* attention (the age-bias fix).
- **Aayush:** LastAttention, Chameleon, Imitation, DAgger.
- **Krishna:** HitLRU, AdaptiveWindow.
- **Kaushal:** HARP (now with an optional protected recent window).

**Fairness.**
- Every policy is listed once in `registry()`. Every test loops over that list, on the same sequences, budgets and seeds.
- **Equal tuning (the `tune` step).** No policy runs with numbers someone picked by hand on a small test. Every policy with settings tries at most 8 of them, its own default first, on the same DEV sequences and train budgets the learned policies are trained on. The best setting is chosen separately for each leave-one-text-type-out pool, exactly like the learned policies, so no policy is tuned on the text type it is tested on.
- **TEST sequences** (odd index) are never used for tuning or training. All results are reported on them.
- **Changed policies:** the grids also contain the original rules (cumulative score for H2O / Hybrid / HeavyHitter-only, no protection for HARP), so the DEV data, not us, decides which version runs.
- **Settings that do nothing:** the tune step flags any setting that never changes a single DEV result (as happened with HitLRU's window).
- **Old results:** every result row records the settings it ran with. Rows computed with other settings are recomputed automatically.
- Offline policies are references and cannot win the verdict.

**Tests.**

| # | Test |
|---|---|
| T1 | competitive ratio: pure text, unseen budgets, spliced topic-shift text, gap breakdown, agreement with the optimum |
| T2 | 6 stress traces, including Kaushal's shifting heavy hitters |
| T3 | adversarial worst case |
| T4 | overhead |
| T5 | perplexity with a physically pruned cache |
| T6 | needle in a haystack |
| T7 | proxy validity |

**How to run.**
1. Set the options in the next cell.
2. Click **Run All**. Every long step prints timestamped progress (`done/total | elapsed | ETA`) and a heartbeat every `HEARTBEAT` seconds. The same lines go to `OUT/run_log.txt`, so you can `tail -f` it from a terminal.
3. Results are cached under `OUT`. After a crash or a stop, re-run all cells and it continues where it stopped.
4. Run `PRESET = "quick"` first (a smoke test, minutes), then `"full"`. `"long"` (T = 2048) is optional.

**GPU work:** the attention-extraction, perplexity and needle steps run in one worker process per GPU, forked from this notebook. The notebook itself never initialises CUDA, so GPU memory is freed after each step. Use Linux, which is standard on GPU servers.

## Options

In [ ]:
PRESET    = "quick"            # "quick" (smoke test) -> "full" (main results, T=512) -> "long" (T=2048)
OUT       = f"runs/{PRESET}"   # results folder. Re-use it to resume; use a NEW folder to change options.
MODEL     = None               # e.g. "Qwen/Qwen2.5-7B"   (None = Qwen/Qwen2.5-0.5B)
DTYPE     = None               # "float32" (default) or "bfloat16" for big models
ABLATIONS = False              # True: Chameleon/HitLRU variants also go through every test (slower)
SYNTHETIC = False              # True: synthetic attention, no model (checks the CPU pipeline only)
N_GPUS    = None               # GPUs to use (None = all visible)
WORKERS   = None               # CPU processes for CPU steps (None = all cores)
OVERRIDES = {}                 # advanced: override any preset value, e.g. {"n_seq": 30, "ppl_seqs": 10}
INSTALL   = True               # pip-install the requirements (torch must already be installed)
HEARTBEAT = 60                 # seconds between "still working" messages during long steps

## Setup: install, imports, progress logging, parallel helpers

In [ ]:
import subprocess, sys
REQUIREMENTS = ['transformers==4.46.3', 'datasets==3.1.0', 'ortools==9.12.4544', 'numpy', 'scipy', 'pandas', 'scikit-learn', 'matplotlib']
if INSTALL:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS], check=True)
    print("installed:", ", ".join(REQUIREMENTS))

In [ ]:
import copy, glob, itertools, json, os, pickle, queue, time, tracemalloc, zlib
import multiprocessing as mp
from dataclasses import dataclass
from typing import Callable
import numpy as np
import pandas as pd
from IPython.display import Markdown, Image, display
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["USE_TF"] = "0"; os.environ["USE_TORCH"] = "1"

FORK = "fork" in mp.get_all_start_methods()
LOG_FILE = [None]


def log(msg):
    """Timestamped line in the notebook and in OUT/run_log.txt."""
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    if LOG_FILE[0]:
        with open(LOG_FILE[0], "a") as f:
            f.write(line + "\n")


def fmt_t(s):
    s = int(max(s, 0))
    return f"{s // 3600}h{s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m{s % 60:02d}s"


class Progress:
    """done/total, elapsed, ETA; printed at most every `every` seconds (and at the end)."""
    def __init__(self, name, total, every=30):
        self.name, self.total, self.every = name, max(int(total), 1), every
        self.done, self.t0 = 0, time.time(); self.last = self.t0
        log(f"{name}: started ({total} units)")

    def _line(self, note=""):
        el = time.time() - self.t0; rate = self.done / el if el > 0 and self.done else 0
        eta = fmt_t((self.total - self.done) / rate) if rate else "?"
        log(f"{self.name}: {self.done}/{self.total} ({100 * self.done / self.total:.0f}%) | elapsed {fmt_t(el)} | ETA {eta}"
            + (f" | {note}" if note else ""))

    def tick(self, n=1, note=""):
        self.done += n
        if time.time() - self.last >= self.every or self.done >= self.total:
            self.last = time.time(); self._line(note)

    def heartbeat(self, extra=""):
        self.last = time.time(); self._line("still working" + (f" ({extra})" if extra else ""))

    def finish(self):
        log(f"{self.name}: finished in {fmt_t(time.time() - self.t0)}")


def pmap(fn, jobs, name):
    """Parallel map over CPU processes with progress + heartbeat. Order of results is not preserved."""
    prog = Progress(name, len(jobs)); out = []
    if WORKERS <= 1 or len(jobs) <= 1 or not FORK:
        for j in jobs:
            out.append(fn(j)); prog.tick()
    else:
        with mp.get_context("fork").Pool(min(WORKERS, len(jobs))) as pool:
            it = pool.imap_unordered(fn, jobs)
            while len(out) < len(jobs):
                try:
                    out.append(it.next(timeout=HEARTBEAT)); prog.tick()
                except mp.TimeoutError:
                    prog.heartbeat(f"{WORKERS} CPU workers busy")
    prog.finish()
    return out


def detect_gpus():
    vis = os.environ.get("CUDA_VISIBLE_DEVICES")
    if vis is not None:
        return [x.strip() for x in vis.split(",") if x.strip()]
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip()
    except Exception:
        return []
    for l in out.splitlines():
        print("  GPU", l)
    return [l.split(",")[0].strip() for l in out.splitlines() if l.strip()]


def gpu_parallel(name, work, units):
    """Run work(shard, n_shards, report) in one forked process per GPU. report("tick", note) advances the
    progress bar, report("log", text) prints a line. The notebook process itself never touches CUDA."""
    n = len(GPU_IDS); prog = Progress(f"{name} [{n} worker(s)]", units, every=20)
    if not FORK:                                   # e.g. Windows: run here, one device
        work(0, 1, lambda kind, msg="": prog.tick(note=msg) if kind == "tick" else log(msg)); prog.finish(); return
    if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
        raise RuntimeError("CUDA was initialised in the notebook process; restart the kernel and Run All again")
    ctx = mp.get_context("fork"); q = ctx.Queue(); procs = []
    wname = lambda i: f"gpu{GPU_IDS[i]}" if GPU_IDS[i] else "cpu"

    def child(i):
        os.environ["CUDA_VISIBLE_DEVICES"] = GPU_IDS[i]
        try:
            work(i, n, lambda kind, msg="": q.put((kind, i, msg)))
            q.put(("done", i, ""))
        except BaseException:
            import traceback
            q.put(("error", i, traceback.format_exc()))
    try:
        for i in range(n):
            p = ctx.Process(target=child, args=(i,), daemon=True); p.start(); procs.append(p)
        finished, errors = set(), []
        while len(finished) < n:
            try:
                kind, i, msg = q.get(timeout=HEARTBEAT)
            except queue.Empty:
                for i, p in enumerate(procs):
                    if i not in finished and not p.is_alive():
                        finished.add(i); errors.append(f"worker {i} died (exit code {p.exitcode}), e.g. out of memory")
                prog.heartbeat(f"{n - len(finished)} worker(s) running"); continue
            if kind == "tick":
                prog.tick(note=f"{wname(i)}: {msg}")
            elif kind == "log":
                log(f"{name} | {wname(i)}: {msg}")
            elif kind == "done":
                finished.add(i)
            elif kind == "error":
                finished.add(i); errors.append(f"worker {i}:\n{msg}")
    finally:
        for p in procs:
            if p.is_alive():
                p.terminate()
            p.join()
    prog.finish()
    if errors:
        raise RuntimeError(f"{name} failed:\n" + "\n".join(errors))


WORKERS = WORKERS or os.cpu_count() or 1
GPU_IDS = detect_gpus()
if N_GPUS:
    GPU_IDS = GPU_IDS[:N_GPUS]
if not GPU_IDS:
    GPU_IDS = [""]
    if not SYNTHETIC:
        print("WARNING: no GPU found - model steps will run on CPU (very slow)")
try:
    import ortools; print("ortools", ortools.__version__, "- fast exact optimum")
except ImportError:
    print("WARNING: ortools missing - slow LP fallback for the optimum (install ortools, Python 3.10-3.12)")
print(f"python {sys.version.split()[0]} | CPU workers {WORKERS} | GPU workers {len(GPU_IDS)} {GPU_IDS} | fork available: {FORK}")

## 0. Configuration presets
`quick` is a smoke test. `full` gives the main results: T = 512, 6 text types and 24 sequences per type, of which half are TEST. `long` uses T = 2048. Budgets are cache sizes as a fraction of T. Budgets that are not in `train_budgets` are *unseen* by the learned policies.

In [ ]:
ALL_TYPES = ["wikipedia", "code", "dialogue", "books", "science", "math"]
_FULL = dict(
    model="Qwen/Qwen2.5-0.5B", dtype="float32", T=512, text_types=ALL_TYPES,
    n_seq=24,                                   # per text type; even index = DEV, odd = TEST
    train_budgets=[0.1, 0.2, 0.3, 0.5],
    test_budgets=[0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5],   # 0.05, 0.15, 0.4 = unseen by learned policies
    n_splice=4,                                 # spliced (topic-shift) sequences per ordering
    ppl_seqs=8, ppl_shift_seqs=2, ppl_budgets=[0.1, 0.15, 0.3],
    needle_depths=[0.1, 0.3, 0.5, 0.7, 0.9], needle_reps=8, needle_budgets=[0.1, 0.2, 0.3, 0.5],
    stress_T=512, stress_seeds=[0, 1, 2], stress_budgets=[0.1, 0.2, 0.3],
    adv_T=64, adv_k=16, adv_iters=150, adv_restarts=3,
    dagger_rounds=3, seed=0,
    verdict_min_gain=0.0025,                    # a "win" needs CI lower bound > this (0.25 points of ratio)
)
PRESETS = {
    "full": _FULL,
    "quick": dict(_FULL, T=256, text_types=["wikipedia", "code", "dialogue"], n_seq=4,
                  train_budgets=[0.1, 0.2, 0.3], test_budgets=[0.1, 0.15, 0.2, 0.3], n_splice=1,
                  ppl_seqs=1, ppl_shift_seqs=1, ppl_budgets=[0.15],
                  needle_depths=[0.1, 0.5, 0.9], needle_reps=2, needle_budgets=[0.2, 0.5],
                  stress_T=256, stress_seeds=[0], adv_T=48, adv_k=8, adv_iters=30, adv_restarts=1,
                  dagger_rounds=1),
    "long": dict(_FULL, T=2048, n_seq=12, train_budgets=[0.1, 0.2, 0.3], test_budgets=[0.05, 0.1, 0.2, 0.3],
                 n_splice=2, ppl_seqs=3, ppl_shift_seqs=1, ppl_budgets=[0.1, 0.2],
                 needle_reps=4, needle_budgets=[0.1, 0.2, 0.3]),
}
SINKS = 4   # used only by the gap breakdown (which tokens count as "sink"), not by any policy decision


def bkey(b):
    return f"{b:.3f}"

## 1. Problem core
`A[t, j]` is the attention of query `t` on key `j`, averaged over layers and heads (rows sum to 1). The cache holds at most `k` tokens, token `t` is always inserted, and eviction is permanent. The reward at step `t` is the attention on cached tokens. Every policy implements `reset`, `select_evictions` (it may only use the past) and `observe`.

In [ ]:
import time

class Policy:
    """Base class. Online policies only see attention of steps < t (via observe)."""
    name = "base"
    online = True
    uses_attention = False

    def reset(self, T, k):
        self.T, self.k = T, k

    def select_evictions(self, t, cache):
        """cache: list of token ids (includes t). Return ids to evict (never t)."""
        raise NotImplementedError

    def observe(self, t, cache, attn):
        """attn[i] = attention of step t on cache[i], renormalised over the cache."""
        pass

### 1a. Base: online policies
Random, SlidingWindow (FIFO), StreamingLLM (sinks + recency), HeavyHitter-only / H2O (heavy hitters, H2O also protects the newest tokens), Hybrid (sinks + newest tokens + heavy hitters).

**The age-bias fix.** The original H2O / Hybrid rank tokens by *cumulative* attention. That favours old tokens, which had more steps to collect attention, so a token that has just left the protected window looks weak and is evicted while it is still being read. On the quick run, 95% of Hybrid's evictions were younger than k/2. The heavy-hitter policies now rank by **average attention per step alive** (`rule='avg'`). On the quick run's DEV sequences, HeavyHitter-only went from 0.646 to 0.980. The original rule (`rule='sum'`) stays in the tuning grid, so the DEV data confirms the choice.

In [ ]:
# ---------------------------------------------------------------- Base: online policies
class RandomPolicy(Policy):
    name = "Random"

    def __init__(self, seed=0):
        self.seed = seed

    def reset(self, T, k):
        super().reset(T, k); self.rng = np.random.default_rng(self.seed)

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        past = cache[:-1]
        return [past[self.rng.integers(len(past))]]


class SlidingWindow(Policy):
    """Keep the k most recent tokens (FIFO). O(1)."""
    name = "SlidingWindow"

    def select_evictions(self, t, cache):
        return [cache[0]] if len(cache) > self.k else []


class StreamingLLM(Policy):
    """Keep the first `sinks` tokens (attention sinks) + the most recent. O(1)."""
    name = "StreamingLLM"

    def __init__(self, sinks=4):
        self.sinks = sinks

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        for j in cache:
            if j >= self.sinks:
                return [j]
        return [cache[0]]


def _hh_key(score, rule, t):
    """Ranking key for heavy-hitter policies. 'sum' = cumulative attention (original H2O rule). 'avg' = average
    attention per step alive = cumulative / age. The sum favours OLD tokens (they had more steps to collect
    attention), so with a small protected window it evicts tokens that have just left the window while they are
    still being read. Quick run, DEV sequences: HeavyHitter-only 0.646 with 'sum' vs 0.980 with 'avg'."""
    if rule == "sum":
        return lambda j: score[j]
    return lambda j: score[j] / max(t - j, 1)


class HeavyHitter(Policy):
    """H2O-style heavy hitters: protect the recent_frac*k newest tokens, evict the lowest-ranked of the rest
    (rule 'avg' by default, 'sum' = original H2O). recent_frac = 0 -> 'HeavyHitter-only'. O(k)."""
    uses_attention = True

    def __init__(self, recent_frac=0.5, rule="avg", name="H2O"):
        assert rule in ("sum", "avg")
        self.recent_frac, self.rule, self.name = recent_frac, rule, name

    def reset(self, T, k):
        super().reset(T, k); self.score = np.zeros(T)

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        r = max(1, int(self.recent_frac * self.k))
        cand = cache[:len(cache) - r] or cache[:-1]
        return [min(cand, key=_hh_key(self.score, self.rule, t))]

    def observe(self, t, cache, attn):
        self.score[np.asarray(cache)] += attn


class Hybrid(Policy):
    """Sinks + recent window + heavy hitters (StreamingLLM U H2O), rule 'avg' by default ('sum' = the original
    Hybrid, which lost 0.907 vs 0.973 on the quick run because of the age bias of the sum). O(k)."""
    name = "Hybrid"
    uses_attention = True

    def __init__(self, sinks=4, recent_frac=0.25, rule="avg"):
        assert rule in ("sum", "avg")
        self.sinks, self.recent_frac, self.rule = sinks, recent_frac, rule

    def reset(self, T, k):
        super().reset(T, k); self.score = np.zeros(T)

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        r = max(1, int(self.recent_frac * self.k))
        cand = [j for j in cache[:len(cache) - r] if j >= self.sinks]
        if not cand:
            cand = [j for j in cache[:-1] if j >= self.sinks] or cache[:-1]
        return [min(cand, key=_hh_key(self.score, self.rule, t))]

    def observe(self, t, cache, attn):
        self.score[np.asarray(cache)] += attn

### 1b. Aayush: LastAttention (TOVA-style) and Chameleon
LastAttention evicts the token with the least attention at the previous step.

Chameleon keeps 5 experts (recency, cumulative, last-step, slow-EMA, hit-rate) and re-weights them online with **Fixed-Share Hedge**. Its decision is the lowest weighted-average rank. Its regret bound covers the weights, measured on a surrogate loss.

In [ ]:
# ---------------------------------------------------------------- Aayush: LastAttention (TOVA-style)
class LastAttention(Policy):
    """Evict the token with the least attention at the most recent step (Oren et al. 2024). O(k).
    sinks: the first `sinks` tokens are never evicted (0 = the original rule)."""
    name = "LastAttention"
    uses_attention = True

    def __init__(self, sinks=0):
        self.sinks = sinks

    def reset(self, T, k):
        super().reset(T, k); self.last = np.zeros(T)

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        cand = [j for j in cache[:-1] if j >= self.sinks] or cache[:-1]
        return [min(cand, key=lambda j: self.last[j])]

    def observe(self, t, cache, attn):
        self.last[np.asarray(cache)] = attn


# ---------------------------------------------------------------- Aayush: Chameleon (Fixed-Share Hedge over 5 experts)
EXPERTS = ["recency", "cumulative", "last-step", "slow-EMA", "hit-rate"]


def _pct(x):
    """Percentile rank in [0,1] (ties broken by position: older = lower)."""
    r = np.empty(len(x)); r[np.argsort(x, kind="stable")] = np.arange(len(x))
    return r / max(len(x) - 1, 1)


class Chameleon(Policy):
    """Fixed-Share-Hedge mixture of five eviction experts; evict the lowest weighted-average rank.
    Regret of the WEIGHTS vs the best fixed expert, on the surrogate loss of _step_losses:
        <= ln(N)/eta + eta*n/8 + n*ln(1/(1-alpha))/eta.     O(N k log k) per step, N = 5.
    eta_scale: eta = eta_scale*sqrt(8 ln N / T); share: alpha = share/T; sinks/grace: protected tokens;
    q_frac: loss looks at the q_frac*k tokens an expert would evict first; lag_frac: score rankings from
    lag steps ago; learn=False: uniform weights; sharp: decision weights w**sharp; experts: subset."""
    uses_attention = True

    def __init__(self, eta_scale=1.0, share=5.0, sinks=4, grace=1 / 16, q_frac=0.25, lag_frac=0.0,
                 learn=True, experts=None, sharp=1.0, name="Chameleon"):
        self.eta_scale, self.share, self.sinks, self.grace = eta_scale, share, sinks, grace
        self.q_frac, self.lag_frac, self.learn, self.sharp, self.name = q_frac, lag_frac, learn, sharp, name
        self.sel = list(range(len(EXPERTS))) if experts is None else list(experts)
        self.N = len(self.sel)

    def reset(self, T, k):
        super().reset(T, k)
        self.cum = np.zeros(T); self.ef = np.zeros(T); self.es = np.zeros(T); self.hits = np.zeros(T)
        self.eta = self.eta_scale * np.sqrt(8 * np.log(max(self.N, 2)) / T)
        self.alpha = min(0.5, self.share / T)
        self.w = np.full(self.N, 1.0 / self.N)
        self.q = max(1, int(self.q_frac * k))
        self.lag = int(round(self.lag_frac * k))
        self.snap = {}
        self.w_hist, self.loss_hist, self.mix_loss = [], [], []

    def _scores(self, state, t, c):
        cum, ef, es, hits = state
        age = np.maximum(t - c, 1)
        allsc = [c.astype(float), cum[c], ef[c], es[c], hits[c] / age]
        return [allsc[i] for i in self.sel]

    def _state(self):
        return (self.cum, self.ef, self.es, self.hits)

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        g = max(2, int(self.grace * self.k))
        pool = cache[:len(cache) - g] if len(cache) - g > 0 else cache[:-1]
        cand = [j for j in pool if j >= self.sinks] or [j for j in cache[:-1] if j >= self.sinks] or cache[:-1]
        c = np.asarray(cand)
        wd = self.w ** self.sharp; wd = wd / wd.sum()
        agg = sum(wi * _pct(s) for wi, s in zip(wd, self._scores(self._state(), t, c)))
        return [cand[int(np.argmin(agg))]]

    def _step_losses(self, t, cache, attn):
        lag = self.lag
        old = self._state() if lag == 0 else self.snap.get(t - lag)
        if old is None:
            return None
        c = np.asarray(cache); a = np.asarray(attn)
        ok = (c >= self.sinks) & (c <= t - 1 - lag)
        if ok.sum() < self.q + 2:
            return None
        c, a = c[ok], a[ok]
        if lag > 0:
            a = self.cum[c] - old[0][c]
        q = self.q; s = np.sort(a)
        lo, hi = s[:q].sum(), s[-q:].sum()
        if hi - lo < 1e-12:
            return None
        loss = np.empty(self.N)
        for i, sc in enumerate(self._scores(old, t - lag, c)):
            bottom = np.argsort(_pct(sc), kind="stable")[:q]
            loss[i] = (a[bottom].sum() - lo) / (hi - lo)
        return np.clip(loss, 0.0, 1.0)

    def observe(self, t, cache, attn):
        c = np.asarray(cache); a = np.asarray(attn)
        if self.lag > 0:
            self.snap[t] = tuple(x.copy() for x in self._state()); self.snap.pop(t - self.lag - 1, None)
        loss = self._step_losses(t, cache, attn) if (self.learn and len(cache) >= self.k) else None
        if loss is not None:
            self.mix_loss.append(float(self.w @ loss)); self.loss_hist.append(loss)
            v = self.w * np.exp(-self.eta * loss); v /= v.sum()
            self.w = (1 - self.alpha) * v + self.alpha / self.N
        self.w_hist.append(self.w.copy())
        self.cum[c] += a
        self.ef[c] = a
        self.es[c] = 0.9 * self.es[c] + 0.1 * a
        self.hits[c] += (a * len(c) >= 1.0)

    def regret_bound(self):
        n = len(self.loss_hist)
        return np.log(self.N) / self.eta + self.eta * n / 8 + n * np.log(1 / (1 - self.alpha)) / self.eta

    def regret(self):
        L = np.sum(self.loss_hist, axis=0)
        return float(np.sum(self.mix_loss) - L.min()), int(np.argmin(L))

### 1c. Krishna: HitLRU and AdaptiveWindow
HitLRU is LRU over attention *hits*: a hit is attention ≥ θ/|cache|. AdaptiveWindow is HitLRU whose protected recent window is learned from where attention lands.

In [ ]:
# ---------------------------------------------------------------- Krishna: paging-style hybrids
class HitLRU(Policy):
    """Sinks + recent window + attention-LRU: token j is 'hit' at step t if attn >= theta/|cache|.
    Evict the token whose last hit is oldest (+ lam*log(1+hits) bonus). O(k)."""
    uses_attention = True

    def __init__(self, sinks=4, recent_frac=0.2, theta=2.0, lam=0.0, name="HitLRU"):
        self.sinks, self.recent_frac, self.theta, self.lam, self.name = sinks, recent_frac, theta, lam, name

    def reset(self, T, k):
        super().reset(T, k); self.last = np.arange(T, dtype=float); self.hits = np.zeros(T)

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        r = max(1, int(self.recent_frac * self.k))
        cand = [j for j in cache[:len(cache) - r] if j >= self.sinks]
        if not cand:
            cand = [j for j in cache[:-1] if j >= self.sinks] or cache[:-1]
        c = np.asarray(cand); pr = self.last[c] + self.lam * np.log1p(self.hits[c])
        return [int(c[np.argmin(pr)])]

    def observe(self, t, cache, attn):
        i = np.asarray(cache); h = attn >= self.theta / len(i)
        self.last[i[h]] = t; self.hits[i[h]] += 1


class AdaptiveWindow(Policy):
    """HitLRU whose recent window r adapts: r = q-quantile of an EMA histogram of attention distance
    (clipped to [lo*k, hi*k]). Note: distances are observed only for cached tokens (feedback bias). O(k + T)."""
    uses_attention = True

    def __init__(self, sinks=4, q=0.9, theta=2.0, beta=0.98, lo=0.05, hi=0.6, name="AdaptiveWindow"):
        self.sinks, self.q, self.theta, self.beta, self.lo, self.hi, self.name = sinks, q, theta, beta, lo, hi, name

    def reset(self, T, k):
        super().reset(T, k); self.last = np.arange(T, dtype=float); self.hist = np.zeros(T); self.r = max(1, int(.25 * k))

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        cand = [j for j in cache[:len(cache) - self.r] if j >= self.sinks]
        if not cand:
            cand = [j for j in cache[:-1] if j >= self.sinks] or cache[:-1]
        c = np.asarray(cand)
        return [int(c[np.argmin(self.last[c])])]

    def observe(self, t, cache, attn):
        i = np.asarray(cache); d = t - i; ns = i >= self.sinks
        self.hist *= self.beta
        np.add.at(self.hist, d[ns], attn[ns])
        tot = self.hist.sum()
        if tot > 0:
            cdf = np.cumsum(self.hist) / tot; dq = int(np.searchsorted(cdf, self.q)) + 1
            self.r = max(1, int(np.clip(dq, self.lo * self.k, self.hi * self.k)))
        h = attn >= self.theta / len(i); self.last[i[h]] = t

### 1d. Kaushal: HARP
Past tokens compete on the percentile ranks of four signals: cumulative attention, EWMA attention, attention trend and peak attention, with fixed weights.

Kaushal's original HARP protects no sinks and no recent window. That suits his synthetic shifting-heavy-hitters test, but on real text it drops recent tokens that are still being read (+226% perplexity in the quick run). HARP now protects 4 sinks + the newest k/4 tokens by default. The original (no protection) stays in the tuning grid.

In [ ]:
# ---------------------------------------------------------------- Kaushal: HARP
class HARP(Policy):
    """HARP: Hybrid Adaptive Retention Policy. Past tokens compete on four causal signals (cumulative attention,
    EWMA recent attention, attention trend = EWMA change, peak attention). Each signal becomes a within-candidate
    percentile rank (ties averaged); evict the lowest weighted rank. O(k) per step.
    recent_frac / sinks: protect the newest recent_frac*k tokens and the first `sinks` tokens. Kaushal's original
    HARP protects nothing (recent_frac=0, sinks=0); on real text that drops recent tokens that are still being
    read (quick run: +226% perplexity), so the default now protects 4 sinks + k/4 recent tokens."""
    name = "HARP"
    uses_attention = True

    def __init__(self, decay=0.90, weights=(0.35, 0.35, 0.20, 0.10), recent_frac=0.25, sinks=4, name="HARP"):
        self.decay, self.name, self.recent_frac, self.sinks = float(decay), name, recent_frac, sinks
        self.weights = np.asarray(weights, dtype=np.float64)
        if not np.isclose(self.weights.sum(), 1.0) or np.any(self.weights < 0):
            raise ValueError("HARP weights must be non-negative and sum to 1")

    def reset(self, T, k):
        super().reset(T, k)
        self.long_score = np.zeros(T); self.ewma_score = np.zeros(T)
        self.previous_ewma = np.zeros(T); self.peak_score = np.zeros(T)

    @staticmethod
    def _percentile_rank(values):
        """Average-tie percentile rank in [0,1] (same values as Kaushal's loop, vectorised)."""
        from scipy.stats import rankdata
        values = np.asarray(values, dtype=np.float64); n = len(values)
        if n <= 1:
            return np.zeros(n)
        return (rankdata(values, method="average") - 1) / (n - 1)

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        if self.recent_frac > 0 or self.sinks > 0:
            r = max(1, int(self.recent_frac * self.k))
            cand = [j for j in cache[:len(cache) - r] if j >= self.sinks] or [j for j in cache[:-1] if j >= self.sinks] or cache[:-1]
        else:
            cand = cache[:-1]
        sig = [self.long_score, self.ewma_score, self.ewma_score - self.previous_ewma, self.peak_score]
        combined = self.weights @ np.vstack([self._percentile_rank(x[cand]) for x in sig])
        return [cand[int(np.argmin(combined))]]

    def observe(self, t, cache, attn):
        ids = np.asarray(cache, dtype=int); v = np.asarray(attn, dtype=np.float64)
        self.previous_ewma[ids] = self.ewma_score[ids]
        self.ewma_score[ids] = self.decay * self.ewma_score[ids] + (1.0 - self.decay) * v
        self.long_score[ids] += v
        self.peak_score[ids] = np.maximum(self.peak_score[ids], v)

### 1e. Offline references, the exact optimum (min-cost flow), the simulator
**Min-cost flow:** time nodes `0..T` plus one node per token.
- Idle arcs `t→t+1` have capacity `k` and cost 0.
- Exit arcs `tok_j → e+1` have capacity 1 and cost `C − V(j,e)`.
- Node 0 supplies `k`, node `T` absorbs `k`, and each time node `j` demands one unit that `tok_j` supplies.

The cut between `t` and `t+1` carries exactly `k` units, so at most `k` tokens are cached. Total unimodularity makes the optimal flow integral, so it is a valid schedule. There are O(T²) arcs. Belady-greedy (least future attention) is **not** optimal, because eviction is permanent.

In [ ]:
# ---------------------------------------------------------------- Base: offline (they know the future)
class BeladyGreedy(Policy):
    """Evict the token with the least FUTURE attention sum_{s>=t} A[s, j]. Not optimal (permanent eviction)."""
    name = "Belady-Greedy (offline)"
    online = False

    def __init__(self, A):
        self.F = np.flip(np.cumsum(np.flip(A, 0), 0), 0)

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        return [min(cache[:-1], key=lambda j: self.F[t, j])]


class ScheduledPolicy(Policy):
    """Replays a schedule: token j is kept during steps j..end[j]."""
    online = False

    def __init__(self, end, name="Optimal (offline)"):
        self.end, self.name = np.asarray(end), name

    def select_evictions(self, t, cache):
        return [j for j in cache if j != t and self.end[j] < t]


class FullCache(Policy):
    name = "Full cache"

    def select_evictions(self, t, cache):
        return []


# ---------------------------------------------------------------- exact offline optimum (min-cost flow)
def optimal_schedule(A, k, scale=1_000_000):
    """EXACT offline optimum by reduction to min-cost flow (OR-Tools).
    Time nodes 0..T + one node per token. Arcs: t->t+1 cap k cost 0 (empty slot);
    tok_j -> e+1 cap 1 cost C - V(j,e) (token j kept during j..e), V(j,e) = sum_{s=j..e} A[s,j].
    Supply +k at 0, -k at T; node j demands 1, tok_j supplies 1 (forces every token in).
    Cut t|t+1 carries k units -> capacity respected; one exit arc per token -> min cost = max attention;
    totally unimodular -> integral. O(T^2) arcs. Returns end[]."""
    from ortools.graph.python import min_cost_flow
    T = A.shape[0]
    colcum = np.cumsum(A, axis=0)
    C = int(scale * (T + 1))
    j_idx, e_idx = np.triu_indices(T)                   # all pairs e >= j
    v = colcum[e_idx, j_idx]                            # = V(j, e) since A[s, j] = 0 for s < j
    tails = np.concatenate([np.arange(T), T + 1 + j_idx])
    heads = np.concatenate([np.arange(1, T + 1), e_idx + 1])
    caps = np.concatenate([np.full(T, k), np.ones(len(j_idx), dtype=np.int64)])
    costs = np.concatenate([np.zeros(T, dtype=np.int64), C - np.round(scale * v).astype(np.int64)])
    smcf = min_cost_flow.SimpleMinCostFlow()
    arcs = smcf.add_arcs_with_capacity_and_unit_cost(tails, heads, caps, costs)
    sup = np.zeros(2 * T + 1, dtype=np.int64)
    sup[0] += k; sup[T] -= k; sup[:T] -= 1; sup[T + 1:] += 1
    smcf.set_nodes_supplies(np.arange(2 * T + 1), sup)
    status = smcf.solve()
    if status != smcf.OPTIMAL:
        raise RuntimeError(f"min-cost flow failed: status {status}")
    flows = smcf.flows(arcs)[T:]
    end = np.full(T, -1)
    nz = np.nonzero(flows)[0]
    end[j_idx[nz]] = e_idx[nz]
    assert (end >= np.arange(T)).all(), "every token must be scheduled"
    return end


def _lp_optimum(A, k):
    """Same network solved as an LP with HiGHS (fallback when OR-Tools is missing; slow for large T)."""
    from scipy.optimize import linprog
    from scipy.sparse import coo_matrix
    T = A.shape[0]; col = np.cumsum(A, axis=0); C = float(T + 1)
    j_idx, e_idx = np.triu_indices(T)
    tails = np.r_[np.arange(T), T + 1 + j_idx]; heads = np.r_[np.arange(1, T + 1), e_idx + 1]
    cost = np.r_[np.zeros(T), C - col[e_idx, j_idx]]; cap = np.r_[np.full(T, k), np.ones(len(j_idx))]
    na = len(tails)
    B = coo_matrix((np.r_[np.ones(na), -np.ones(na)], (np.r_[tails, heads], np.r_[np.arange(na), np.arange(na)])),
                   shape=(2 * T + 1, na)).tocsr()
    sup = np.zeros(2 * T + 1); sup[0] += k; sup[T] -= k; sup[:T] -= 1; sup[T + 1:] += 1
    r = linprog(cost, A_eq=B, b_eq=sup, bounds=np.c_[np.zeros(na), cap], method="highs-ds")
    assert r.status == 0, r.message
    x = np.round(r.x[T:]).astype(int)
    end = np.full(T, -1); nz = np.nonzero(x)[0]; end[j_idx[nz]] = e_idx[nz]
    assert (end >= np.arange(T)).all(); return end


_WARNED = [False]


def get_optimum(A, k):
    try:
        return optimal_schedule(A, k)
    except ImportError:
        if not _WARNED[0]:
            print("  [warn] ortools not installed: using the LP fallback (much slower for T > 256)", file=sys.stderr)
            _WARNED[0] = True
        return _lp_optimum(A, k)


# ---------------------------------------------------------------- simulator
def simulate(A, k, policy, record=False):
    """Run `policy` on A with capacity k. Returns (per-step retained attention, mean decision seconds,
    occupancy matrix M[t, j] = 1 if j cached at step t, or None)."""
    T = A.shape[0]
    policy.reset(T, k)
    cache, mass = [], np.zeros(T)
    M = np.zeros((T, T), dtype=bool) if record else None
    tt = 0.0
    for t in range(T):
        cache.append(t)
        t0 = time.perf_counter()
        ev = policy.select_evictions(t, cache)
        tt += time.perf_counter() - t0
        if ev:
            evs = set(ev)
            assert t not in evs, "cannot evict the current token"
            cache = [j for j in cache if j not in evs]
        assert len(cache) <= k, f"{policy.name} exceeded capacity at step {t}"
        if record:
            M[t, cache] = True
        row = A[t, cache]
        mass[t] = row.sum()
        t0 = time.perf_counter()
        policy.observe(t, cache, row / max(row.sum(), 1e-12))
        tt += time.perf_counter() - t0
    return mass, tt / T, M


def teacher_forced(A, k, end, policy):
    """Hand the policy the OPTIMUM's cache every step; how often does it drop a token the optimum drops?
    Returns (agreement, chance level)."""
    T = A.shape[0]; policy.reset(T, k); cache = []; hit = n = 0; chance = 0.0
    for t in range(T):
        cache.append(t)
        opt_ev = {j for j in cache[:-1] if end[j] < t}
        if len(cache) > k and opt_ev:
            ev = policy.select_evictions(t, cache)
            hit += bool(ev) and ev[0] in opt_ev; n += 1; chance += len(opt_ev) / (len(cache) - 1)
        cache = [j for j in cache if j not in opt_ev]
        row = A[t, cache]; policy.observe(t, cache, row / max(row.sum(), 1e-12))
    return hit / max(n, 1), chance / max(n, 1)


def synthetic_attention(T, seed=0, sinks=2, n_heavy=8, decay=0.15):
    """Random attention with sinks, recency bias and a few re-read 'heavy' tokens. Testing only."""
    rng = np.random.default_rng(seed)
    heavy = rng.choice(np.arange(sinks, T), size=min(n_heavy, T - sinks), replace=False)
    A = np.zeros((T, T))
    for t in range(T):
        j = np.arange(t + 1)
        logits = -decay * (t - j) + rng.normal(0, 1.0, t + 1)
        logits[j < sinks] += 3.0
        logits[np.isin(j, heavy)] += 2.0
        w = np.exp(logits - logits.max())
        A[t, :t + 1] = w / w.sum()
    return A

## 2. Aayush: learned policies (Imitation, DAgger)
The teacher is the exact optimum. The student is a logistic regression on 8 features that use only the past. DAgger also retrains on the states the student itself visits, relabelled by the optimum.

In [ ]:
FEATURES = ["age/k", "recent (age<k/4)", "avg attention", "score vs cache mean",
            "last-step attention", "EMA attention", "is sink (j<4)", "budget k/T"]
EMA_ALPHA = 0.3
MAX_ROWS = 1_500_000


class Tracker:
    """Past-only per-token statistics."""
    def reset(self, T, k):
        self.T, self.k = T, k
        self.score = np.zeros(T); self.last = np.zeros(T); self.ema = np.zeros(T)

    def update(self, cache, attn):
        c = np.asarray(cache)
        self.score[c] += attn; self.last[c] = attn
        self.ema[c] = (1 - EMA_ALPHA) * self.ema[c] + EMA_ALPHA * attn

    def features(self, t, cand):
        c = np.asarray(cand); k = self.k; age = t - c; sc = self.score[c]
        return np.column_stack([age / k, (age < k / 4).astype(float), sc / np.maximum(age, 1) * k,
                                sc / (sc.mean() + 1e-12), self.last[c] * k, self.ema[c] * k,
                                (c < 4).astype(float), np.full(len(c), k / self.T)])


class ImitationScorer(Policy):
    """Evict argmin of the learned keep-probability (logistic regression on past-only features). O(k)."""
    uses_attention = True

    def __init__(self, w, b, mu, sd, name="Imitation"):
        self.w, self.b, self.mu, self.sd, self.name = w, b, mu, sd, name
        self.tr = Tracker()

    def reset(self, T, k):
        super().reset(T, k); self.tr.reset(T, k)

    def select_evictions(self, t, cache):
        if len(cache) <= self.k:
            return []
        cand = cache[:-1]
        z = ((self.tr.features(t, cand) - self.mu) / self.sd) @ self.w + self.b
        return [cand[int(np.argmin(z))]]

    def observe(self, t, cache, attn):
        self.tr.update(cache, attn * len(cache))


def collect(A, k, end, base, rng, per_step=16, h_frac=0.5):
    """Roll out `base`; at each eviction step record features of sampled cached tokens, label
    y = 1 if the optimum still holds the token at t + h_frac*k."""
    T = A.shape[0]; H = max(1, int(h_frac * k))
    tr = Tracker(); tr.reset(T, k); base.reset(T, k)
    cache, X, Y = [], [], []
    for t in range(T):
        cache.append(t)
        if len(cache) > k:
            cand = np.asarray(cache[:-1])
            pick = cand if len(cand) <= per_step else rng.choice(cand, per_step, replace=False)
            X.append(tr.features(t, pick)); Y.append((end[pick] >= t + H).astype(int))
        ev = set(base.select_evictions(t, cache))
        cache = [j for j in cache if j not in ev]
        row = A[t, cache]; a = row / max(row.sum(), 1e-12)
        base.observe(t, cache, a); tr.update(cache, a * len(cache))
    return np.vstack(X), np.concatenate(Y)


def fit_student(X, Y, name, seed=0):
    from sklearn.linear_model import LogisticRegression
    if len(Y) > MAX_ROWS:
        idx = np.random.default_rng(seed).choice(len(Y), MAX_ROWS, replace=False); X, Y = X[idx], Y[idx]
    mu, sd = X.mean(0), X.std(0) + 1e-9
    clf = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced").fit((X - mu) / sd, Y)
    pol = ImitationScorer(clf.coef_[0], float(clf.intercept_[0]), mu, sd, name)
    pol.n_train, pol.pos_rate = len(Y), float(Y.mean())
    return pol


def _teacher_data(items, rng, per_step=16, h_frac=0.5):
    Xs, Ys = [], []
    for A, k, end in items:
        # roll-out policies that produce training states (kept at the original rules: trained students stay valid)
        for base in [ScheduledPolicy(end), HeavyHitter(0.5, "sum", "H2O"), Hybrid(rule="sum"), SlidingWindow()]:
            X, Y = collect(A, k, end, base, rng, per_step, h_frac); Xs.append(X); Ys.append(Y)
    return Xs, Ys


def train_imitation(items, seed=0, name="Imitation", h_frac=0.5):
    rng = np.random.default_rng(seed)
    Xs, Ys = _teacher_data(items, rng, 16, h_frac)
    return fit_student(np.vstack(Xs), np.concatenate(Ys), name, seed)


def dagger_train(items, rounds=3, seed=0, name="DAgger", h_frac=0.5, per_step=8):
    """DAgger (Ross et al. 2011): round 0 = imitation; each round adds the states the CURRENT student
    visits, relabelled by the optimum, and refits. Returns [round 0, ..., rounds]."""
    rng = np.random.default_rng(seed)
    Xs, Ys = _teacher_data(items, rng, per_step, h_frac)
    students = [fit_student(np.vstack(Xs), np.concatenate(Ys), f"{name} r0", seed)]
    for r in range(1, rounds + 1):
        for A, k, end in items:
            X, Y = collect(A, k, end, students[-1], rng, per_step, h_frac); Xs.append(X); Ys.append(Y)
        students.append(fit_student(np.vstack(Xs), np.concatenate(Ys), f"{name} r{r}", seed))
    return students


def state_auc(student, items, own, seed=0, h_frac=0.5):
    """Ranking AUC of the student on the optimum's states (own=False) or on its own states (own=True)."""
    from sklearn.metrics import roc_auc_score
    rng = np.random.default_rng(seed); Xs, Ys = [], []
    for A, k, end in items:
        X, Y = collect(A, k, end, copy.deepcopy(student) if own else ScheduledPolicy(end), rng, 16, h_frac)
        Xs.append(X); Ys.append(Y)
    X, Y = np.vstack(Xs), np.concatenate(Ys)
    if len(set(Y.tolist())) < 2:
        return float("nan")
    return float(roc_auc_score(Y, ((X - student.mu) / student.sd) @ student.w + student.b))

## 3. The registry: the single list of policies that every test iterates over
Each `Entry` also holds the policy's **tuning grid**: at most `TUNE_BUDGET` = 8 settings, its default first. The `tune` step picks one setting per pool on DEV data. `legacy` names the settings that reproduce the version used before tuning existed, so old results made with exactly those settings are reused.

To add a new algorithm, add one `Entry` line here. Give it a grid if it has settings.

In [ ]:
@dataclass
class Ctx:
    """What a factory may use. Online/learned policies must ignore A and end (checked by the causality test).
    tuned: {policy name: settings} chosen on DEV data for this instance's pool (None = each policy's default)."""
    A: np.ndarray = None
    k: int = 0
    end: np.ndarray = None
    students: dict = None
    tuned: dict = None


TUNE_BUDGET = 8   # FAIRNESS: every tunable policy may try at most this many settings, all on the same DEV data


def _grid(**axes):
    """Cartesian grid; the first value of every axis is the policy's default, so grid[0] = the default."""
    keys = list(axes)
    return [dict(zip(keys, v)) for v in itertools.product(*axes.values())]


@dataclass
class Entry:
    name: str
    family: str            # "online" | "learned" | "offline" | "ablation"
    factory: Callable      # factory(ctx, **settings) -> Policy
    author: str = ""
    grid: list = None      # settings tried by the 'tune' stage (grid[0] = default); None = nothing to tune
    legacy: dict = None    # settings that reproduce the pre-tuning version exactly (lets old results be reused)

    def params(self, ctx):
        if not self.grid:
            return {}
        if ctx.tuned is None:
            return dict(self.grid[0])
        if self.name not in ctx.tuned:
            raise RuntimeError(f"no tuned settings for '{self.name}' - run the 'tune' stage")
        return dict(ctx.tuned[self.name])

    def make(self, ctx, params=None):
        return self.factory(ctx, **(self.params(ctx) if params is None else params))


def fingerprint(entry, tuned_all):
    """The exact settings a policy ran with, over all pools. Saved with every result row, so rows computed with
    different settings are detected and recomputed. '-' = nothing to tune, or identical to the pre-tuning version."""
    if not entry.grid:
        return "-"
    used = {p: v.get(entry.name) for p, v in sorted(tuned_all.items())}
    if entry.legacy is not None and all(u == entry.legacy for u in used.values()):
        return "-"
    return "c%08x" % zlib.crc32(json.dumps(used, sort_keys=True).encode())


def grid_id(entry):
    return "g%08x" % zlib.crc32(json.dumps(entry.grid, sort_keys=True).encode()) if entry.grid else "-"


def _student(c, nm):
    if not c.students or nm not in c.students:
        raise RuntimeError(f"no trained student '{nm}' for this pool - run the 'train' stage")
    s = copy.deepcopy(c.students[nm]); s.name = nm
    return s


def registry(ablations=False):
    """Every policy, with its tuning grid. FAIRNESS: grids have at most TUNE_BUDGET settings each; the policy's
    own default comes first and wins ties. Grids include the original rule where a policy was changed (H2O,
    Hybrid, HeavyHitter-only: rule 'sum'; HARP: no protection), so DEV data, not us, decides which version runs."""
    R = [
        Entry("Random", "online", lambda c: RandomPolicy(), "Base"),
        Entry("SlidingWindow", "online", lambda c: SlidingWindow(), "Base"),
        Entry("StreamingLLM", "online", lambda c, **p: StreamingLLM(**p), "Base",
              _grid(sinks=[4, 1, 2, 8]), dict(sinks=4)),
        Entry("HeavyHitter-only", "online", lambda c, **p: HeavyHitter(0.0, name="HeavyHitter-only", **p), "Base",
              _grid(rule=["avg", "sum"]), dict(rule="sum")),
        Entry("H2O", "online", lambda c, **p: HeavyHitter(name="H2O", **p), "Base",
              _grid(rule=["avg", "sum"], recent_frac=[0.5, 0.1, 0.25, 0.75]), dict(rule="sum", recent_frac=0.5)),
        Entry("Hybrid", "online", lambda c, **p: Hybrid(**p), "Base",
              _grid(rule=["avg", "sum"], recent_frac=[0.25, 0.1, 0.5, 0.75]), dict(rule="sum", recent_frac=0.25)),
        Entry("LastAttention", "online", lambda c, **p: LastAttention(**p), "Aayush",
              _grid(sinks=[0, 4]), dict(sinks=0)),
        Entry("Chameleon", "online", lambda c, **p: Chameleon(**p), "Aayush",
              _grid(eta_scale=[1.0, 0.3, 3.0, 10.0], share=[5.0, 1.0]), dict(eta_scale=1.0, share=5.0)),
        Entry("HitLRU", "online", lambda c, **p: HitLRU(**p), "Krishna",
              _grid(theta=[2.0, 0.5, 1.0, 4.0], recent_frac=[0.2, 0.5]), dict(theta=2.0, recent_frac=0.2)),
        Entry("AdaptiveWindow", "online", lambda c, **p: AdaptiveWindow(**p), "Krishna",
              _grid(theta=[2.0, 0.5, 1.0, 4.0], q=[0.9, 0.8]), dict(theta=2.0, q=0.9)),
        Entry("HARP", "online", lambda c, protect, decay: HARP(decay=decay, recent_frac=protect, sinks=4 if protect > 0 else 0),
              "Kaushal", _grid(protect=[0.25, 0.1, 0.5, 0.0], decay=[0.9, 0.7]), dict(protect=0.0, decay=0.9)),
        Entry("Imitation", "learned", lambda c: _student(c, "Imitation"), "Aayush"),
        Entry("DAgger", "learned", lambda c: _student(c, "DAgger"), "Aayush"),
        Entry("Belady-Greedy (offline)", "offline", lambda c: BeladyGreedy(c.A), "Base"),
        Entry("Optimal (offline)", "offline", lambda c: ScheduledPolicy(c.end), "Base"),
    ]
    if ablations:
        A = [("Chameleon: equal weights", dict(learn=False)), ("Chameleon: no fixed-share", dict(share=0.0)),
             ("Chameleon: no sinks", dict(sinks=0)), ("Chameleon: no grace", dict(grace=0.0)),
             ("Chameleon: eta x3", dict(eta_scale=3.0)), ("Chameleon: follow-the-leader", dict(sharp=8.0)),
             ("Chameleon: lagged loss k/8", dict(lag_frac=0.125))]
        A += [(f"Chameleon: without {e}", dict(experts=[j for j in range(len(EXPERTS)) if j != i]))
              for i, e in enumerate(EXPERTS)]
        A += [("HitLRU: lam=1", "hitlru")]
        for nm, kw in A:
            if kw == "hitlru":
                R.append(Entry(nm, "ablation", lambda c, nm=nm: HitLRU(lam=1.0, name=nm), "Krishna"))
            else:
                R.append(Entry(nm, "ablation", lambda c, nm=nm, kw=kw: Chameleon(name=nm, **kw), "Aayush"))
    for e in R:
        assert not e.grid or len(e.grid) <= TUNE_BUDGET, f"{e.name}: tuning grid larger than TUNE_BUDGET"
    return R


def needs_future(e):
    return e.family == "offline"

## 4. Test generators: stress traces (6 patterns, including Kaushal's shifting heavy hitters) and the adversary
The adversary hill-climbs over attention logits to find the input on which a policy does worst against the optimum. The lowest ratio it finds is an empirical upper bound on that policy's competitive ratio.

In [ ]:
def stress_traces(T, seed=0):
    """Six hand-built attention patterns (the last one is Kaushal's shifting-heavy-hitters test)."""
    rng = np.random.default_rng(seed)

    def recency(decay=0.25):
        A = np.zeros((T, T))
        for t in range(T):
            A[t, :t + 1] = np.exp(-decay * (t - np.arange(t + 1)))
        return A / A.sum(1, keepdims=True)

    def blend(base, target, w):
        return (1 - w[:, None]) * base + w[:, None] * target

    def finish(A):
        A = np.tril(A * np.exp(rng.normal(0, .15, A.shape))); return A / A.sum(1, keepdims=True)

    base = recency(); tr = {}; steps = np.arange(T)
    tr["pure recency"] = finish(base)
    sk = np.zeros((T, T)); sk[:, :4] = .25
    tr["sinks + recency"] = finish(blend(base, np.tril(sk), np.full(T, .8)))
    heavy = np.arange(20, T - 20, 40); tgt = np.zeros((T, T))
    for t in range(T):
        h = heavy[heavy < t]
        if len(h):
            tgt[t, h] = 1 / len(h)
    tr["periodic re-reads"] = finish(blend(base, tgt, np.where(steps % 32 < 4, .7, 0.)))
    h1, h2 = np.arange(10, 26), np.arange(T // 2 - 40, T // 2 - 24); tgt = np.zeros((T, T))
    tgt[:T // 2, h1] = 1 / len(h1); tgt[T // 2:, h2] = 1 / len(h2)
    tr["sudden topic change"] = finish(blend(base, np.tril(tgt), np.full(T, .5)))
    late = np.arange(T // 4, T // 4 + 4); tgt = np.zeros((T, T)); tgt[:, late] = .25
    tr["unpredictable late burst"] = finish(blend(base, np.tril(tgt), np.where(steps >= T - 16, .6, 0.)))
    tr["shifting heavy hitters"] = shifting_heavy_hitters(T, [11, 23, 37][seed % 3] + 100 * (seed // 3))
    return tr


def shifting_heavy_hitters(T, seed):
    """Kaushal's diagnostic: one heavy-hitter group (tokens 8..15) is replaced by another (T/2..T/2+7)
    halfway through, on top of noise and a mild recency bias. Seeds 11, 23, 37 as in the HARP notebook."""
    rng = np.random.default_rng(seed); A = np.zeros((T, T))
    group_a = np.arange(8, min(16, T)); group_b = np.arange(max(8, T // 2), min(T // 2 + 8, T))
    for t in range(T):
        j = np.arange(t + 1)
        logits = rng.normal(0.0, 0.65, size=t + 1)
        logits[np.isin(j, group_a if t < T // 2 else group_b)] += 3.0
        logits -= 0.025 * (t - j)
        w = np.exp(logits - logits.max()); A[t, :t + 1] = w / w.sum()
    return A


def _softmax_rows(L):
    T = L.shape[0]; A = np.zeros((T, T))
    for t in range(T):
        z = L[t, :t + 1]; w = np.exp(z - z.max()); A[t, :t + 1] = w / w.sum()
    return A


def ratio_on(A, k, entry, students, tuned=None):
    end = get_optimum(A, k)
    om = simulate(A, k, ScheduledPolicy(end))[0][k:].mean()
    pol = entry.make(Ctx(A=A, k=k, end=end, students=students, tuned=tuned))
    return simulate(A, k, pol)[0][k:].mean() / om


def adversary(entry, students, T=64, k=16, iters=150, restarts=3, seed=0, tuned=None):
    """Hill-climb over attention logits to FIND an input where the policy is worst vs the optimum.
    The lowest ratio found is an empirical upper bound on the policy's competitive ratio."""
    rng = np.random.default_rng(seed); best = 2.0
    for _ in range(restarts):
        A0 = synthetic_attention(T, int(rng.integers(1_000_000)), sinks=int(rng.integers(0, 3)),
                                 n_heavy=int(rng.integers(2, 8)))
        L = np.log(A0 + 1e-9); cur = ratio_on(_softmax_rows(L), k, entry, students, tuned)
        for _ in range(iters):
            L2 = L.copy(); kind = rng.integers(3)
            if kind == 0:
                a = int(rng.integers(0, T - 4)); L2[a:a + 4] += rng.normal(0, 1.5, (4, T))
            elif kind == 1:
                j = int(rng.integers(0, T - 8)); a = int(rng.integers(j, T - 2)); b = min(T, a + int(rng.integers(2, 16)))
                L2[a:b, j] += rng.choice([-3.0, 3.0]) * rng.random() * 2
            else:
                t_ = int(rng.integers(1, T)); L2[t_, :t_ + 1] += rng.normal(0, 2.0, t_ + 1)
            val = ratio_on(_softmax_rows(L2), k, entry, students, tuned)
            if val < cur:
                L, cur = L2, val
        best = min(best, cur)
    return best

## 5. Self-tests
These check that:
- the optimum equals brute force, and the LP fallback gives the same value;
- no policy beats the optimum, and every policy respects the capacity;
- **no online policy uses the future** (future attention rows are replaced by random ones, and past decisions must not change);
- Chameleon's regret stays within its bound.

If this cell fails, do not trust any later number.

In [ ]:
def check_causal(entry, A, k, end, students, t0=None, seed=0, params=None):
    """Decisions up to t0 must not change when the attention rows after t0 are replaced by random ones."""
    T = A.shape[0]; t0 = T // 2 if t0 is None else t0
    A2 = A.copy(); r = np.random.default_rng(seed)
    for t in range(t0 + 1, T):
        w = r.random(t + 1) ** 4; A2[t, :t + 1] = w / w.sum()
    m1 = simulate(A, k, entry.make(Ctx(A=A, k=k, end=end, students=students), params))[0]
    m2 = simulate(A2, k, entry.make(Ctx(A=A2, k=k, end=end, students=students), params))[0]
    return bool(np.allclose(m1[:t0 + 1], m2[:t0 + 1], atol=1e-12))


def _brute(A, k):
    T = A.shape[0]; col = np.cumsum(A, 0); best = [-1.0]

    def rec(j, ends):
        if j == T:
            best[0] = max(best[0], sum(col[ends[i], i] for i in range(T))); return
        for e in range(j, T):
            ends.append(e)
            if all(sum(1 for i in range(j + 1) if i <= s <= ends[i]) <= k for s in range(T)):
                rec(j + 1, ends)
            ends.pop()
    rec(0, []); return best[0]


def selftest(ablations=True):
    print("self-tests")
    for seed in range(6):
        A = synthetic_attention(7, seed, sinks=1, n_heavy=2); k = 2 + seed % 3
        e = get_optimum(A, k); tot = sum(np.cumsum(A, 0)[e[j], j] for j in range(7))
        assert abs(tot - _brute(A, k)) < 1e-6, f"optimum != brute force (seed {seed})"
        e2 = _lp_optimum(A, k); tot2 = sum(np.cumsum(A, 0)[e2[j], j] for j in range(7))
        assert abs(tot2 - tot) < 1e-6, "LP fallback != min-cost flow"
    print("  [1] optimum == brute force == LP fallback on 6 small instances")
    tr = [(synthetic_attention(96, s), int(b * 96)) for s in (10, 11) for b in (0.2, 0.3)]
    items = [(A, k, get_optimum(A, k)) for A, k in tr]
    st = dict(Imitation=train_imitation(items), DAgger=dagger_train(items, rounds=1)[-1])
    print("  [2] learned policies train on synthetic data")
    A = synthetic_attention(120, 3); k = 24; end = get_optimum(A, k)
    om = simulate(A, k, ScheduledPolicy(end))[0][k:].sum()
    R = registry(ablations=ablations); n_set = 0
    for e in R:
        for p in (e.grid or [None]):                                  # EVERY setting the tuner may pick
            pol = e.make(Ctx(A=A, k=k, end=end, students=st), p); n_set += 1
            m = simulate(A, k, pol)[0][k:].sum()                      # simulate() asserts capacity <= k
            assert m <= om + 1e-9, f"{e.name} {p} beats the optimum?!"
            if not needs_future(e):
                assert check_causal(e, A, k, end, st, params=p), f"{e.name} {p} uses future information"
    print(f"  [3] {len(R)} policies, {n_set} settings: capacity respected, none beats the optimum, no online policy sees the future")
    p = Chameleon(); simulate(A, k, p); reg, _ = p.regret()
    assert reg <= p.regret_bound() + 1e-9, "Chameleon regret exceeds its bound"
    print(f"  [4] Chameleon regret {reg:.2f} <= bound {p.regret_bound():.2f}")
    print("self-tests passed")


selftest(ablations=ABLATIONS)

## 6. Model side: texts, attention extraction (memory-light hooks), decoding with a physically pruned cache

In [ ]:
def _hf_text(name, config, split, fn, need, cap, sep="\n\n"):
    from datasets import load_dataset
    ds = load_dataset(name, config, split=split, streaming=True)
    parts, n = [], 0
    for ex in ds:
        s = fn(ex)
        if not s:
            continue
        s = s[:cap]; parts.append(s); n += len(s)
        if n >= need:
            break
    return sep.join(parts)


def _stdlib_code(need, cap):
    import glob
    import sysconfig
    root = sysconfig.get_paths()["stdlib"]
    files = sorted(glob.glob(os.path.join(root, "*.py"))) + sorted(glob.glob(os.path.join(root, "*", "*.py")))
    files = [f for f in files if "test" not in f.lower() and "site-packages" not in f and "idlelib" not in f]
    parts, n = [], 0
    for f in files:
        try:
            s = open(f, encoding="utf-8").read()[:cap]
        except Exception:
            continue
        if len(s) < 2000:
            continue
        parts.append(s); n += len(s)
        if n >= need:
            break
    return "\n\n".join(parts)


TEXT_LOADERS = {
    "wikipedia": lambda need, cap: _hf_text("Salesforce/wikitext", "wikitext-103-raw-v1", "test",
                                            lambda e: e["text"], need, cap, sep=""),
    "code": _stdlib_code,
    "dialogue": lambda need, cap: _hf_text("HuggingFaceH4/ultrachat_200k", None, "test_sft",
                                           lambda e: "\n".join(f"{m['role'].capitalize()}: {m['content']}" for m in e["messages"]),
                                           need, cap),
    "books": lambda need, cap: _hf_text("emozilla/pg19-test", None, "test", lambda e: e["text"], need, cap),
    "science": lambda need, cap: _hf_text("ccdv/arxiv-summarization", "document", "test", lambda e: e["article"], need, cap),
    "math": lambda need, cap: _hf_text("openai/gsm8k", "main", "test",
                                       lambda e: f"Question: {e['question']}\nAnswer: {e['answer']}", need, cap),
}


def load_model(cfg):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(cfg["model"])
    model = AutoModelForCausalLM.from_pretrained(cfg["model"], torch_dtype=getattr(torch, cfg["dtype"]),
                                                 attn_implementation="eager").to(dev).eval()
    return tok, model


def attention_matrix(model, ids):
    """One forward pass with the full cache. Returns A (T x T, mean over layers and heads, rows sum to 1)
    and nll[t] (loss predicting ids[t+1]). Per-layer attention is summed on the fly by hooks and then
    discarded, so memory is O(T^2) instead of O(layers * heads * T^2)."""
    import torch
    import torch.nn.functional as F
    dev = model.device; T = len(ids)
    acc = torch.zeros(T, T, device=dev, dtype=torch.float32); n = [0]
    mods = [m for nm, m in model.named_modules() if nm.endswith("self_attn")]

    def hook(mod, args, out):
        if isinstance(out, tuple) and len(out) > 1 and out[1] is not None:
            acc.add_(out[1][0].float().mean(0)); n[0] += 1
            return (out[0], None) + tuple(out[2:])
    hs = [m.register_forward_hook(hook) for m in mods]
    try:
        with torch.no_grad():
            x = torch.as_tensor(ids, dtype=torch.long, device=dev)[None]
            out = model(x, output_attentions=True, use_cache=False)
            nll = F.cross_entropy(out.logits[0, :-1].float(), x[0, 1:], reduction="none")
    finally:
        for h in hs:
            h.remove()
    if n[0] == 0:
        raise RuntimeError("no attention weights captured: load the model with attn_implementation='eager'")
    A = (acc / n[0]).cpu().numpy().astype(np.float64)
    A = np.tril(A); A /= A.sum(1, keepdims=True)
    return A, nll.cpu().numpy()


def _prune(past, keep_idx):
    if isinstance(past, tuple):
        return tuple((k[:, :, keep_idx], v[:, :, keep_idx]) for k, v in past)
    if hasattr(past, "key_cache"):
        for i in range(len(past.key_cache)):
            past.key_cache[i] = past.key_cache[i][:, :, keep_idx]
            past.value_cache[i] = past.value_cache[i][:, :, keep_idx]
        if hasattr(past, "_seen_tokens"):
            past._seen_tokens = past.key_cache[0].shape[2]
        return past
    for layer in past.layers:
        layer.keys = layer.keys[:, :, keep_idx]; layer.values = layer.values[:, :, keep_idx]
    return past


def run_with_eviction(model, ids, k, policy, gen_tokens=0):
    """Decode token by token, physically deleting evicted KV entries (kept tokens keep their original
    RoPE positions). Returns nll[t] (predicting ids[t+1]) and an optional greedy continuation."""
    import torch
    import torch.nn.functional as F
    from transformers import DynamicCache
    dev = model.device; ids = torch.as_tensor(ids, dtype=torch.long); T = len(ids)
    policy.reset(T, k); need = policy.uses_attention
    cache, past = [], DynamicCache(); nll = np.full(T - 1, np.nan); logits = None
    with torch.no_grad():
        for t in range(T):
            cache.append(t)
            ev = policy.select_evictions(t, cache)
            if ev:
                evs = set(ev)
                keep = [i for i, j in enumerate(cache[:-1]) if j not in evs]
                past = _prune(past, torch.tensor(keep, device=dev))
                cache = [j for j in cache if j not in evs]
            assert len(cache) <= k
            out = model(ids[t:t + 1][None].to(dev), past_key_values=past,
                        position_ids=torch.tensor([[t]], device=dev),
                        attention_mask=torch.ones(1, len(cache), device=dev, dtype=torch.long),
                        use_cache=True, output_attentions=need)
            past = out.past_key_values; logits = out.logits[0, -1].float()
            if t < T - 1:
                nll[t] = -F.log_softmax(logits, -1)[ids[t + 1]].item()
            if need:
                a = torch.stack([x[0, :, -1].float().mean(0) for x in out.attentions]).mean(0).cpu().numpy()
                policy.observe(t, cache, a / a.sum())
        gen, pos = [], T
        for _ in range(gen_tokens):
            nxt = int(logits.argmax()); gen.append(nxt)
            out = model(torch.tensor([[nxt]], device=dev), past_key_values=past,
                        position_ids=torch.tensor([[pos]], device=dev),
                        attention_mask=torch.ones(1, len(cache) + len(gen), device=dev, dtype=torch.long),
                        use_cache=True)
            past, logits = out.past_key_values, out.logits[0, -1].float(); pos += 1
    return nll, gen


def model_selftest(tok, model, cfg):
    """Hook-extracted attention is causal & normalised; decoding with a full cache reproduces the
    full forward pass; decoding with eviction respects capacity."""
    ids = tok("The quick brown fox jumps over the lazy dog. " * 8, add_special_tokens=False)["input_ids"][:64]
    A, nll = attention_matrix(model, ids)
    assert np.allclose(A.sum(1), 1) and np.allclose(np.triu(A, 1), 0), "attention matrix malformed"
    nll2, _ = run_with_eviction(model, ids, len(ids), FullCache())
    tol = 1e-2 if cfg["dtype"] == "float32" else 0.25
    diff = float(np.max(np.abs(nll - nll2)))
    assert diff < tol, f"pruned-cache decoding does not match the full forward pass (max diff {diff:.4f})"
    run_with_eviction(model, ids, 16, HeavyHitter(0.5, "sum", "H2O"))
    return diff

## 7. Storage and per-job workers
Every result is written to `OUT` as soon as it is computed, which is what makes a run resumable.

In [ ]:
class Store:
    def __init__(self, out):
        self.out = out
        for d in ["data/attn", "opt", "students", "tuned", "results"]:
            os.makedirs(os.path.join(out, d), exist_ok=True)

    def p(self, *a):
        return os.path.join(self.out, *a)

    def cfg(self):
        return json.load(open(self.p("config.json")))

    def meta(self):
        return json.load(open(self.p("data/meta.json")))

    def attn_path(self, key):
        return self.p("data/attn", f"{key}.npz")

    def load_attn(self, key):
        z = np.load(self.attn_path(key))
        return z["A"].astype(np.float64), z["nll"].astype(np.float64), z["ids"]

    def opt_path(self, key, b):
        return self.p("opt", f"{key}__{bkey(b)}.npz")

    def load_opt(self, key, b):
        z = np.load(self.opt_path(key, b)); return z["end"], float(z["optm"])

    def tuned_all(self):
        f = self.p("tuned", "params.json")
        if not os.path.exists(f):
            raise RuntimeError("no tuned settings - run the 'tune' stage first")
        return json.load(open(f))

    def tuned(self, pool):
        return self.tuned_all()[pool]

    def fids(self, ablations=False):
        ta = self.tuned_all(); return {e.name: fingerprint(e, ta) for e in registry(ablations)}

    def students(self, pool):
        f = self.p("students", f"{pool}.pkl")
        return pickle.load(open(f, "rb")) if os.path.exists(f) else None


def atomic_write(path, text):
    tmp = path + f".tmp{os.getpid()}"
    with open(tmp, "w") as f:
        f.write(text)
    os.replace(tmp, path)


def _done_set(path, cols):
    import pandas as pd
    if not os.path.exists(path):
        return set()
    d = pd.read_csv(path)
    return set(map(tuple, d[cols].astype(str).values.tolist()))


def _append_rows(path, rows):
    import pandas as pd
    pd.DataFrame(rows).to_csv(path, mode="a", header=not os.path.exists(path), index=False)


def pool_of(inst):
    """Training pool of the learned policy used on this instance: never contains its own text type."""
    return f"no-{inst['text']}" if inst["kind"] == "pure" else "all"


def _opt_job(job):
    out, key, b = job; S = Store(out)
    if os.path.exists(S.opt_path(key, b)):
        return 0
    A = S.load_attn(key)[0]; k = int(b * A.shape[0])
    end = get_optimum(A, k); optm = simulate(A, k, ScheduledPolicy(end))[0][k:].mean()
    np.savez(S.opt_path(key, b), end=end.astype(np.int32), optm=optm)
    return 1


def _items(S, insts, budgets):
    out = []
    for d in insts:
        A = S.load_attn(d["key"])[0]
        for b in budgets:
            out.append((A, int(b * A.shape[0]), S.load_opt(d["key"], b)[0]))
    return out


def _train_job(job):
    out, pool, keys, rounds, budgets, seed = job; S = Store(out)
    f = S.p("students", f"{pool}.pkl")
    if os.path.exists(f):
        return pool
    insts = [d for d in S.meta()["instances"] if d["key"] in set(keys)]
    items = _items(S, insts, budgets)
    imi = train_imitation(items, seed=seed)
    dg = dagger_train(items, rounds=rounds, seed=seed)
    pickle.dump(dict(Imitation=imi, DAgger=dg[-1], DAgger_rounds=dg, train_keys=keys), open(f, "wb"))
    return pool


def _ratio_job(job):
    """T1 on one instance: every policy x every budget -> ratio, gap breakdown, teacher-forced agreement."""
    out, inst, budgets, ablations, only = (tuple(job) + (None,))[:5]; S = Store(out)
    A = S.load_attn(inst["key"])[0]; T = A.shape[0]; st = S.students(pool_of(inst))
    tu = S.tuned(pool_of(inst)); FID = S.fids(ablations)
    Tm, Jm = np.indices((T, T))
    rows = []
    for b in budgets:
        k = int(b * T); end, optm = S.load_opt(inst["key"], b)
        Mo = simulate(A, k, ScheduledPolicy(end), record=True)[2]
        SINK = Jm < SINKS; REC = ((Tm - Jm) < k / 4) & ~SINK; MID = ~(SINK | REC)
        for e in [x for x in registry(ablations) if only is None or x.name in only]:
            ctx = Ctx(A=A, k=k, end=end, students=st, tuned=tu)
            mass, dt, M = simulate(A, k, e.make(ctx), record=True)
            d = (A * (Mo.astype(float) - M))[k:]
            agree, chance = teacher_forced(A, k, end, e.make(Ctx(A=A, k=k, end=end, students=st, tuned=tu)))
            rows.append(dict(key=inst["key"], text=inst["text"], kind=inst["kind"], seq=inst["seq"], split=inst["split"],
                             budget=b, policy=e.name, family=e.family, ratio=mass[k:].mean() / optm,
                             lost_sink=(d * SINK[k:]).sum(1).mean() / optm, lost_recent=(d * REC[k:]).sum(1).mean() / optm,
                             lost_longrange=(d * MID[k:]).sum(1).mean() / optm, agree=agree, chance=chance,
                             us_per_step=dt * 1e6, cfg_id=FID[e.name]))
    return rows


def _tune_job(job):
    """DEV tuning on one sequence: every setting in the grid of every tunable policy, at the train budgets."""
    out, inst, budgets, ablations, only = job; S = Store(out); A = S.load_attn(inst["key"])[0]; T = A.shape[0]; rows = []
    for b in budgets:
        k = int(b * T); end, optm = S.load_opt(inst["key"], b); ctx = Ctx(A=A, k=k, end=end)
        for e in registry(ablations):
            if not e.grid or e.name not in only:
                continue
            for ci, p in enumerate(e.grid):
                m = simulate(A, k, e.make(ctx, p))[0][k:].mean()
                rows.append(dict(key=inst["key"], text=inst["text"], budget=b, policy=e.name, config=ci,
                                 params=json.dumps(p, sort_keys=True), ratio=m / optm))
    return rows


def _stress_job(job):
    out, T, seed, budgets, ablations, only = (tuple(job) + (None,))[:6]; S = Store(out); st = S.students("all"); rows = []
    tu = S.tuned("all"); FID = S.fids(ablations)
    for tname, A in stress_traces(T, seed).items():
        for b in budgets:
            k = int(b * T); end = get_optimum(A, k); om = simulate(A, k, ScheduledPolicy(end))[0][k:].mean()
            for e in [x for x in registry(ablations) if only is None or x.name in only]:
                m = simulate(A, k, e.make(Ctx(A=A, k=k, end=end, students=st, tuned=tu)))[0][k:].mean()
                rows.append(dict(trace=tname, budget=b, seed=seed, policy=e.name, family=e.family, ratio=m / om,
                                 cfg_id=FID[e.name]))
    return rows


def _adv_job(job):
    out, name, cfg, ablations = job; S = Store(out)
    e = [x for x in registry(ablations) if x.name == name][0]
    w = adversary(e, S.students("all"), cfg["adv_T"], cfg["adv_k"], cfg["adv_iters"], cfg["adv_restarts"], cfg["seed"],
                  tuned=S.tuned("all"))
    return dict(policy=name, family=e.family, worst_ratio=w, cfg_id=S.fids(ablations)[name])


def _diag_job(job):
    """Policy-specific diagnostics (not used for ranking): Chameleon regret + weights, DAgger rounds."""
    out, inst, budgets = job; S = Store(out); A = S.load_attn(inst["key"])[0]; T = A.shape[0]; rows, wrows, drows = [], [], []
    cham = S.tuned(pool_of(inst))["Chameleon"]; fid = S.fids()["Chameleon"]       # the tuned Chameleon, as tested
    for b in budgets:
        k = int(b * T); p = Chameleon(**cham); simulate(A, k, p)
        if len(p.loss_hist) >= 5:
            reg, best = p.regret()
            rows.append(dict(key=inst["key"], text=inst["text"], budget=b, regret=reg, bound=p.regret_bound(),
                             best_fixed_expert=EXPERTS[best], holds=reg <= p.regret_bound() + 1e-9, cfg_id=fid))
        if inst["seq"] <= 1:
            W = np.array(p.w_hist)
            for t in range(0, len(W), max(1, len(W) // 64)):
                wrows.append(dict(key=inst["key"], text=inst["text"], budget=b, step=t, **{x: W[t, i] for i, x in enumerate(EXPERTS)}))
    st = S.students(pool_of(inst))
    if st and inst["kind"] == "pure":
        for b in budgets:
            k = int(b * T); end, optm = S.load_opt(inst["key"], b)
            for r, stu in enumerate(st["DAgger_rounds"]):
                drows.append(dict(key=inst["key"], text=inst["text"], budget=b, round=r,
                                  ratio=simulate(A, k, copy.deepcopy(stu))[0][k:].mean() / optm))
    return rows, wrows, drows

## 8. Statistics and report

In [ ]:
def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float)
    if len(x) == 0:
        return (np.nan, np.nan)
    r = np.random.default_rng(seed)
    return tuple(np.percentile(r.choice(x, (n, len(x))).mean(1), [2.5, 97.5]))


def wilson(s, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = s / n; den = 1 + z * z / n; c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return max(0.0, c - h), min(1.0, c + h)


def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for i, idx in enumerate(order):
        run = max(run, (m - i) * p[idx]); adj[idx] = min(1.0, run)
    return adj


def leaderboard(df, val="ratio", higher=True):
    """Per policy: mean of per-text means, bootstrap CI over sequences, worst text, p5, mean rank."""
    import pandas as pd
    d = df.copy()
    d["rank"] = d.groupby(["key", "budget"])[val].rank(ascending=not higher, method="average")
    per_seq = d.groupby(["policy", "key"])[val].mean()
    out = {}
    for p, g in d.groupby("policy"):
        tm = g.groupby("text")[val].mean(); lo, hi = boot_ci(per_seq.loc[p].values)
        out[p] = {"mean": tm.mean(), "CI lo": lo, "CI hi": hi,
                  ("worst text" if higher else "worst text (max)"): tm.min() if higher else tm.max(),
                  "p5" if higher else "p95": np.percentile(g[val], 5 if higher else 95),
                  "mean rank": g["rank"].mean(), "n seq": g["key"].nunique()}
    return pd.DataFrame(out).T.astype(float).sort_values("mean", ascending=not higher)


def pairwise(df, val="ratio"):
    import pandas as pd
    piv = df.pivot_table(index=["key", "budget"], columns="policy", values=val)
    P = list(piv.columns); W = pd.DataFrame(0.0, index=P, columns=P)
    for a, b in itertools.permutations(P, 2):
        W.loc[a, b] = (piv[a] > piv[b]).mean() + 0.5 * (piv[a] == piv[b]).mean()
    return W


def vs_leader(df, leader, val="ratio"):
    import pandas as pd
    from scipy import stats
    ps = df.groupby(["policy", "key"])[val].mean().unstack("policy")
    rows = []
    for p in ps.columns:
        if p == leader:
            continue
        d = (ps[leader] - ps[p]).dropna().values
        try:
            pv = stats.wilcoxon(d).pvalue
        except ValueError:
            pv = np.nan
        lo, hi = boot_ci(d)
        rows.append(dict(policy=p, leader_minus_policy=d.mean(), ci_lo=lo, ci_hi=hi, p_wilcoxon=pv, n_seq=len(d)))
    r = pd.DataFrame(rows)
    if len(r):
        r["p_holm"] = holm(r["p_wilcoxon"].fillna(1.0).values)
    return r.set_index("policy")


def choose_params(df, pools, entries):
    """Per pool: the setting with the highest mean DEV ratio over the pool's sequences x train budgets.
    Ties (within 1e-12) go to the lower index, i.e. the policy's own default."""
    out = {}
    for pool, keys in pools.items():
        d = df[df.key.isin(keys)]; out[pool] = {}
        for e in entries:
            m = d[d.policy == e.name].groupby("config").ratio.mean().sort_index()
            out[pool][e.name] = dict(e.grid[int(m.index[int(np.argmax(m.values >= m.max() - 1e-12))])] if len(m) else e.grid[0])
    return out


def tuning_tables(df, entries, tuned):
    """(1) per policy: default vs chosen DEV ratio; (2) per setting: the largest change it causes. A setting
    that never changes a single result on any DEV instance is flagged NO EFFECT (e.g. a dead parameter)."""
    import pandas as pd
    summ, eff = [], []
    for e in entries:
        g = df[df.policy == e.name]
        if not len(g):
            continue
        piv = g.pivot_table(index=["key", "budget"], columns="config", values="ratio"); m = piv.mean()
        ch = tuned["all"][e.name]; ci = e.grid.index(ch)
        pools = sorted({json.dumps(v[e.name], sort_keys=True) for v in tuned.values() if e.name in v})
        summ.append(dict(policy=e.name, settings_tried=len(e.grid), default=json.dumps(e.grid[0]), dev_default=m[0],
                         chosen=json.dumps(ch), dev_chosen=m[ci], dev_worst_setting=m.min(), gain_vs_default=m[ci] - m[0],
                         same_choice_all_pools="yes" if len(pools) == 1 else f"no ({len(pools)} different)"))
        G = pd.DataFrame(e.grid).astype(str)
        for ax in [c for c in G.columns if G[c].nunique() > 1]:
            others = [c for c in G.columns if c != ax]
            groups = list(G.groupby(others).groups.values()) if others else [G.index]
            spread = rng = 0.0; compared = 0
            for idx in groups:
                cols = [c for c in idx if c in piv.columns]
                if len(cols) >= 2:
                    sub = piv[cols]; compared += 1
                    spread = max(spread, float((sub.max(axis=1) - sub.min(axis=1)).max()))
                    rng = max(rng, float(sub.mean().max() - sub.mean().min()))
            eff.append(dict(policy=e.name, setting=ax, values=" / ".join(G[ax].unique()), max_change_one_instance=spread,
                            max_change_mean=rng, verdict="cannot isolate (only varies together with other settings)" if not compared
                            else "NO EFFECT (never changes any result)" if spread < 1e-12 else
                            ("negligible (< 0.001)" if rng < 1e-3 else "matters")))
    return pd.DataFrame(summ), pd.DataFrame(eff)


def current_rows(df, fid):
    """Keep only rows of policies in the registry that ran with the current settings (cfg_id)."""
    if df is None:
        return None
    c = df["cfg_id"].astype(str) if "cfg_id" in df else np.array(["-"] * len(df))
    return df[np.array([fid.get(p) == x for p, x in zip(df.policy, c)], dtype=bool)]


def md(df, fmt="{:.3f}"):
    def f(v):
        if isinstance(v, (float, np.floating)):
            return "" if np.isnan(v) else fmt.format(v)
        return str(v)
    cols = [str(df.index.name or "")] + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for idx, row in df.iterrows():
        lines.append("| " + " | ".join([str(idx)] + [f(v) for v in row.values]) + " |")
    return "\n".join(lines)


def _read(S, pattern):
    import glob

    import pandas as pd
    fs = sorted(glob.glob(S.p("results", pattern)))
    fs = [f for f in fs if os.path.getsize(f) > 1]
    return pd.concat([pd.read_csv(f, dtype={"cfg_id": str}) for f in fs], ignore_index=True) if fs else None


def stage_report(S, cfg):
    import pandas as pd
    from scipy import stats
    res = S.p("results"); out = []; W = out.append
    unseen = [b for b in cfg["test_budgets"] if b not in cfg["train_budgets"]]
    FID = dict(S.fids(cfg.get("ablations", False)), **{"Full cache": "-"})
    r1 = current_rows(_read(S, "t1_ratio.csv"), FID)
    if r1 is None or not len(r1):
        print("report: run the 'cpu' stage first"); return
    r1["unseen"] = r1.budget.isin(unseen)
    test = r1[r1.split == "test"]; pure = test[test.kind == "pure"]; shift = test[test.kind == "shift"]
    online = lambda d: d[d.family.isin(["online", "learned"])]          # contenders: no offline refs, no ablations
    W(f"# KV-cache eviction benchmark - results\n\nModel `{cfg['model']}`, T = {cfg['T']}, text types: "
      f"{', '.join(S.meta()['types'])}{' (SYNTHETIC attention)' if S.meta().get('synthetic') else ''}. "
      f"TEST sequences per pure type: {pure.groupby('text').key.nunique().to_dict()}. Unseen budgets: {unseen}.\n")
    # ---- verdict
    prim = online(pure[pure.unseen]) if len(unseen) else online(pure)
    lb = leaderboard(prim); lead, run = lb.index[0], lb.index[1]
    ps = prim.groupby(["policy", "key"]).ratio.mean().unstack("policy"); diff = (ps[lead] - ps[run]).dropna().values
    lo, hi = boot_ci(diff)
    W("## Verdict (pre-registered rule)\nPrimary metric: competitive ratio on **unseen budgets, pure text, TEST "
      "sequences**, mean of per-text means; offline references excluded. A policy wins only if the lower end of the "
      "paired 95% bootstrap CI of its gap to the runner-up exceeds the practical threshold; otherwise TIE.\n")
    mg = cfg.get("verdict_min_gain", 0.0025)
    call = "CLEAR WIN" if lo > mg else ("TIE (CI contains 0)" if lo <= 0 else f"TIE (gap below the practical threshold {mg})")
    W(f"- **{lead}** {lb.loc[lead, 'mean']:.4f} vs runner-up **{run}** {lb.loc[run, 'mean']:.4f}: paired difference "
      f"{diff.mean():+.4f}, CI [{lo:+.4f}, {hi:+.4f}] -> **{call}**")
    if len(diff) < 10:
        W(f"- WARNING: only {len(diff)} TEST sequences; the bootstrap CI is unreliable below about 10. Use the full preset.")
    wt = lb["worst text"].idxmax(); W(f"- Most robust (best worst-text ratio): {wt} ({lb.loc[wt, 'worst text']:.3f})\n")
    # ---- at a glance
    g = pd.DataFrame(index=[e.name for e in registry(cfg.get("ablations", False))])
    g["family"] = [e.family for e in registry(cfg.get("ablations", False))]
    g["ratio pure"] = leaderboard(pure)["mean"]
    if len(unseen):
        g["ratio unseen"] = leaderboard(pure[pure.unseen])["mean"]
    if len(shift):
        g["ratio shift"] = leaderboard(shift)["mean"]
    g["agree w/ opt"] = pure.groupby("policy").agree.mean()
    st = current_rows(_read(S, "t2_stress.csv"), FID)
    if st is not None:
        g["stress worst"] = st.groupby(["policy", "trace"]).ratio.mean().groupby("policy").min()
    adv = current_rows(_read(S, "t3_adversary.csv"), FID)
    if adv is not None:
        g["adversary worst"] = adv.set_index("policy").worst_ratio
    pp = current_rows(_read(S, "t5_ppl_shard*.csv"), FID)
    if pp is not None:
        g["dPPL% median"] = pp[pp.kind == "pure"].groupby("policy").dppl_pct.median()
    nd = current_rows(_read(S, "t6_needle_shard*.csv"), FID)
    if nd is not None:
        g["needle acc"] = nd[nd.policy != "Full cache"].groupby("policy").success.mean()
    ov = current_rows(_read(S, "t4_overhead.csv"), FID)
    if ov is not None:
        g["us/step"] = ov.set_index("policy").us_per_step
    g = g.sort_values("ratio pure", ascending=False); g.index.name = "policy"
    W("## All tests at a glance\nEvery policy went through every test on identical instances. ratio = retained "
      "attention / optimum (TEST sequences); stress/adversary = worst case found; dPPL% = median perplexity increase "
      "vs full cache; needle = retrieval accuracy over all budgets/depths.\n")
    W(md(g)); W("")
    g.to_csv(os.path.join(res, "summary_at_a_glance.csv"))
    # ---- fair tuning
    ts, te = _read(S, "tuning_summary.csv"), _read(S, "tuning_effects.csv")
    if ts is not None:
        W(f"## Fair tuning (DEV sequences only)\nEvery policy with settings tried at most {TUNE_BUDGET} of them (its own "
          "default first, which wins ties) on the same DEV sequences and train budgets the learned policies are trained "
          "on. The best mean ratio was chosen separately for every leave-one-text-type-out pool, so no policy is tuned "
          "on the text type it is tested on, and TEST sequences are never used. The grids include the original rules of "
          "the changed policies (rule 'sum' for H2O / Hybrid / HeavyHitter-only, no protection for HARP), so the DEV "
          "data decided which version runs. Random and SlidingWindow have nothing to tune; Imitation and DAgger are "
          "trained on the same DEV data. Settings shown: pool 'all'.\n")
        W(md(ts.set_index("policy"), "{:.4f}"))
        if te is not None and len(te):
            W("\n### Does each setting matter? (largest change of the ratio it caused, DEV)\n"); W(md(te.set_index("policy"), "{:.4g}"))
            for _, r in te[te.verdict.str.startswith("NO EFFECT")].iterrows():
                W(f"- WARNING: `{r.setting}` of {r.policy} never changed a single DEV result: that part of the policy "
                  "does nothing on this data.")
        W("")
    # ---- T1 details
    cols = ["mean", "CI lo", "CI hi", "worst text", "p5", "mean rank", "n seq"]
    W("## T1 Competitive ratio\n### Pure text, TEST, all budgets\n"); W(md(leaderboard(pure)[cols]))
    if len(unseen):
        W("\n### Pure text, TEST, unseen budgets (primary)\n"); W(md(leaderboard(pure[pure.unseen])[cols]))
    if len(shift):
        W("\n### Distribution shift: spliced sequences (TEST)\n"); W(md(leaderboard(shift)[cols]))
    t = pure.pivot_table(index="policy", columns="text", values="ratio"); W("\n### Per text type\n"); W(md(t))
    t = pure.pivot_table(index="policy", columns="budget", values="ratio"); W("\n### Per budget\n"); W(md(t))
    v = vs_leader(online(pure), leaderboard(online(pure)).index[0])
    W(f"\n### Every online/learned policy vs the leader ({leaderboard(online(pure)).index[0]}), paired over sequences, Holm-corrected\n")
    W(md(v, "{:.4g}"))
    pairwise(pure).to_csv(os.path.join(res, "t1_pairwise_winrate.csv"))
    dec = pure.groupby("policy")[["lost_sink", "lost_recent", "lost_longrange"]].mean()
    dec["total gap"] = dec.sum(axis=1); dec = dec.sort_values("total gap")
    W("\n### Where the gap to the optimum comes from (share of the optimum's attention lost)\n"); W(md(dec))
    ag = pure.groupby("policy")[["agree", "chance"]].mean().sort_values("agree", ascending=False)
    W("\n### Teacher-forced agreement (given the optimum's cache, drops the same token)\n"); W(md(ag))
    # ---- T2/T3/T4
    if st is not None:
        tab = st.pivot_table(index="policy", columns="trace", values="ratio"); tab["WORST"] = tab.min(axis=1)
        W("\n## T2 Stress traces (ratio)\n"); W(md(tab.sort_values("WORST", ascending=False)))
    if adv is not None:
        W("\n## T3 Adversary (lowest ratio found; lower = easier to break)\n")
        W(md(adv.set_index("policy")[["worst_ratio"]].sort_values("worst_ratio", ascending=False)))
    if ov is not None:
        W("\n## T4 Overhead (Python, same synthetic instance)\n"); W(md(ov.set_index("policy")[["us_per_step", "peak_KB"]], "{:.1f}"))
    # ---- T5 perplexity + T7 proxy validity
    if pp is not None:
        W("\n## T5 Perplexity with a physically pruned cache (TEST sequences)\n")
        for kind in ["pure", "shift"]:
            d = pp[pp.kind == kind]
            if not len(d):
                continue
            lbp = leaderboard(d, "dppl_pct", higher=False)
            W(f"\n### {kind} text: dPPL % (lower is better)\n"); W(md(lbp, "{:.2f}"))
            W("\n" + md(d.pivot_table(index="policy", columns="budget", values="dppl_pct", aggfunc="median"), "{:.2f}")
              + "\n(median dPPL % per budget)")
        taus_all, taus_top = [], []
        m = pp.merge(r1[["key", "budget", "policy", "ratio"]], on=["key", "budget", "policy"])
        for _, s in m.groupby(["key", "budget"]):
            if len(s) >= 3:
                taus_all.append(stats.kendalltau(s.ratio, -s.dppl_pct)[0])
                top = s[s.ratio >= s.ratio.median()]
                if len(top) >= 3:
                    taus_top.append(stats.kendalltau(top.ratio, -top.dppl_pct)[0])
        W("\n## T7 Proxy validity: does attention retained rank policies like perplexity does?\n"
          "Kendall tau per (sequence, budget) between ranking by ratio and by dPPL; +1 = identical ranking.\n")
        for nm, x in [("all policies", taus_all), ("top half by ratio (excludes collapsed policies)", taus_top)]:
            x = np.array([v for v in x if not np.isnan(v)])
            if len(x):
                lo, hi = boot_ci(x); W(f"- {nm}: mean tau {x.mean():+.3f} [{lo:+.3f}, {hi:+.3f}] over {len(x)} settings")
    # ---- T6 needle
    if nd is not None:
        W("\n## T6 Needle in a haystack\n")
        a = nd.groupby("policy").success.agg(["sum", "count"]); a["acc"] = a["sum"] / a["count"]
        ci = [wilson(s, n) for s, n in zip(a["sum"], a["count"])]; a["CI lo"] = [c[0] for c in ci]; a["CI hi"] = [c[1] for c in ci]
        W(md(a.sort_values("acc", ascending=False)[["acc", "CI lo", "CI hi", "count"]]))
        W("\n### Accuracy by budget\n"); W(md(nd.pivot_table(index="policy", columns="budget", values="success")))
        W("\n### Accuracy by depth\n"); W(md(nd.pivot_table(index="policy", columns="depth", values="success")))
    # ---- diagnostics
    rc = _read(S, "diag_chameleon_regret.csv"); dr = _read(S, "diag_dagger_rounds.csv")
    rc = rc[rc.cfg_id.astype(str) == FID["Chameleon"]] if rc is not None and "cfg_id" in rc else None
    W("\n## Diagnostics (policy-specific, not used for ranking)\n")
    if rc is not None:
        W(f"- Chameleon regret bound holds in {rc.holds.mean():.0%} of {len(rc)} runs; median regret/bound = {(rc.regret / rc.bound).median():.2f}")
        W(f"- Best fixed expert in hindsight: {rc.best_fixed_expert.value_counts().to_dict()}")
    if dr is not None and len(dr):
        W("- DAgger ratio by round (held-out text type, TEST): " + ", ".join(f"r{r} {v:.4f}" for r, v in dr.groupby("round").ratio.mean().items()))
    atomic_write(os.path.join(res, "summary.md"), "\n".join(out) + "\n")
    _plots(S, pure, st, nd, pp)
    print(f"report: {os.path.join(res, 'summary.md')}")


def _plots(S, pure, st, nd, pp):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return
    res = S.p("results"); texts = sorted(pure.text.unique())
    fig, axes = plt.subplots(1, len(texts), figsize=(5 * len(texts), 4), sharey=True, squeeze=False)
    for ax, nm in zip(axes[0], texts):
        s = pure[pure.text == nm].groupby(["budget", "policy"]).ratio.mean().unstack()
        for p in s.columns:
            ax.plot(s.index, s[p], marker="o", ms=3, label=p, ls="--" if "offline" in p else "-")
        ax.set_title(nm); ax.set_xlabel("budget"); ax.grid(alpha=.3)
    axes[0][0].set_ylabel("competitive ratio"); axes[0][-1].legend(fontsize=6)
    plt.tight_layout(); plt.savefig(os.path.join(res, "t1_ratio_by_budget.png"), dpi=130); plt.close()
    if st is not None:
        tab = st.pivot_table(index="policy", columns="trace", values="ratio")
        fig, ax = plt.subplots(figsize=(10, 0.4 * len(tab) + 1.5)); im = ax.imshow(tab.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(tab.shape[1])); ax.set_xticklabels(tab.columns, rotation=20, fontsize=8)
        ax.set_yticks(range(tab.shape[0])); ax.set_yticklabels(tab.index, fontsize=8); plt.colorbar(im)
        for i in range(tab.shape[0]):
            for j in range(tab.shape[1]):
                ax.text(j, i, f"{tab.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
        plt.tight_layout(); plt.savefig(os.path.join(res, "t2_stress.png"), dpi=130); plt.close()
    if nd is not None:
        s = nd[nd.policy != "Full cache"].pivot_table(index="budget", columns="policy", values="success")
        ax = s.plot(marker="o", figsize=(8, 4)); ax.set_ylabel("needle accuracy"); ax.legend(fontsize=6)
        plt.tight_layout(); plt.savefig(os.path.join(res, "t6_needle_by_budget.png"), dpi=130); plt.close()
    if pp is not None:
        s = pp[pp.kind == "pure"].groupby("policy").dppl_pct.median().sort_values()
        ax = s.plot.barh(figsize=(7, 0.35 * len(s) + 1.5)); ax.set_xscale("symlog"); ax.set_xlabel("median dPPL % (lower = better)")
        plt.tight_layout(); plt.savefig(os.path.join(res, "t5_ppl.png"), dpi=130); plt.close()

## 9. Stages
Each stage skips work that is already saved and prints progress while it runs.

In [ ]:
def stage_data():
    """Texts -> DEV/TEST sequences, spliced (topic-shift) sequences, needle filler; then attention extraction on GPU."""
    S, cfg = STORE, CFG; T = cfg["T"]; synthetic = cfg["synthetic"]
    if not os.path.exists(S.p("data/meta.json")):
        types = list(cfg["text_types"]); rng = np.random.default_rng(cfg["seed"]); CH = {}
        if synthetic:
            for nm in types:
                CH[nm] = [rng.integers(100, 30000, T) for _ in range(cfg["n_seq"] + 3 * cfg["n_splice"] + 4)]
        else:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained(cfg["model"])
            tok.model_max_length = 10 ** 9   # we tokenize whole texts only to cut them into T-token chunks; silences a harmless warning
            n_need = cfg["n_seq"] + 3 * cfg["n_splice"] + 2
            for nm in types:
                t0 = time.time(); log(f"data: downloading + tokenizing '{nm}' ...")
                extra = (cfg["needle_reps"] * len(cfg["needle_depths"]) + 4) if nm in ("wikipedia", types[0]) else 0
                try:
                    # ~6 chars/token; read 4x more text than needed so the evenly spread chunks span many documents
                    text = TEXT_LOADERS[nm]((n_need + extra) * T * 6 * 4, max(2 * T * 6, 20000))
                except Exception as ex:
                    log(f"data: [skip] text type '{nm}' failed to load: {ex}"); continue
                ids = np.asarray(tok(text, add_special_tokens=False)["input_ids"])
                CH[nm] = [ids[p * T:(p + 1) * T] for p in range(len(ids) // T)]
                log(f"data: {nm}: {len(ids)} tokens -> {len(CH[nm])} chunks of {T} ({fmt_t(time.time() - t0)})")
            types = [t for t in types if t in CH]
            if not types:
                raise RuntimeError("no text type could be loaded (internet / Hugging Face access?)")
        insts, free = [], {}
        for nm in types:                                              # evenly spread main sequences
            ch = CH[nm]; n = min(cfg["n_seq"], max(2, len(ch) - 3 * cfg["n_splice"] - 1))
            pick = sorted(set(np.linspace(0, len(ch) - 1, n).round().astype(int).tolist()))
            free[nm] = [c for i, c in enumerate(ch) if i not in set(pick)] or ch
            for s, p in enumerate(pick):
                insts.append(dict(key=f"{nm}__{s:03d}", text=nm, kind="pure", seq=s,
                                  split="dev" if s % 2 == 0 else "test", ids=ch[p]))
        ptr = {nm: 0 for nm in types}
        if len(types) >= 2:                                           # spliced topic-shift sequences, from unused chunks
            L = min(3, len(types)); SEG = T // L
            for o in range(len(types)):
                order = tuple(types[(o + j) % len(types)] for j in range(L)); tname = "shift:" + ">".join(x[:4] for x in order)
                for i in range(cfg["n_splice"]):
                    parts = []
                    for pos, nm in enumerate(order):
                        c = free[nm][ptr[nm] % len(free[nm])]; ptr[nm] += 1
                        parts.append(c[:SEG if pos < L - 1 else T - SEG * (L - 1)])
                    insts.append(dict(key=f"{tname.replace(':', '_').replace('>', '-')}__{i:03d}", text=tname, kind="shift",
                                      seq=i, split="test", ids=np.concatenate(parts)))
        ftype = "wikipedia" if "wikipedia" in types else types[0]
        if synthetic:
            filler = rng.integers(100, 30000, T * (cfg["needle_reps"] * len(cfg["needle_depths"]) + 2))
        else:
            filler = np.concatenate(free[ftype][ptr[ftype]:] or free[ftype])
        np.savez(S.p("data/ids.npz"), **{d["key"]: np.asarray(d["ids"]) for d in insts})
        np.save(S.p("data/needle_filler.npy"), np.asarray(filler))
        meta = dict(instances=[{k: v for k, v in d.items() if k != "ids"} for d in insts], types=types,
                    filler_type=ftype, synthetic=synthetic)
        atomic_write(S.p("data/meta.json"), json.dumps(meta, indent=1))
        log(f"data: {len(insts)} instances ({sum(d['kind'] == 'pure' for d in insts)} pure, "
            f"{sum(d['kind'] == 'shift' for d in insts)} spliced) over types {types}")
    meta = S.meta(); idz = np.load(S.p("data/ids.npz"))
    todo = [d for d in meta["instances"] if not os.path.exists(S.attn_path(d["key"]))]
    if not todo:
        log(f"data: attention for all {len(meta['instances'])} instances already saved - skipping"); return

    def save(d, A, nll):
        np.savez_compressed(S.attn_path(d["key"]), A=A.astype(np.float32), nll=nll.astype(np.float32), ids=idz[d["key"]])

    if synthetic:
        prog = Progress("data: synthetic attention", len(todo))
        for d in todo:
            ti = meta["types"].index(d["text"]) if d["kind"] == "pure" else len(meta["types"])
            A = synthetic_attention(T, zlib.crc32(d["key"].encode()), sinks=2 + ti % 3, n_heavy=4 + 4 * ti, decay=0.08 + 0.05 * ti)
            save(d, A, np.abs(np.random.default_rng(1).normal(2.5, 1, T - 1))); prog.tick()
        prog.finish(); return

    def work(i, n, report):
        tok, model = load_model(cfg); report("log", f"model loaded on {model.device}")
        if i == 0:
            report("log", f"model self-test passed (full-cache decode vs forward pass: max |dNLL| = {model_selftest(tok, model, cfg):.2e})")
        for d in todo[i::n]:
            t0 = time.time(); A, nll = attention_matrix(model, idz[d["key"]]); save(d, A, nll)
            report("tick", f"{d['key']} ({time.time() - t0:.1f}s)")
    gpu_parallel("data: attention extraction", work, len(todo))


def stage_optimum():
    S, cfg = STORE, CFG
    jobs = [(S.out, d["key"], b) for d in S.meta()["instances"] for b in cfg["test_budgets"] if not os.path.exists(S.opt_path(d["key"], b))]
    if not jobs:
        log("optimum: all schedules already saved - skipping"); return
    pmap(_opt_job, jobs, "optimum: min-cost flow (one unit = one sequence x budget)")


def stage_train():
    S, cfg = STORE, CFG; meta = S.meta()
    dev = [d for d in meta["instances"] if d["kind"] == "pure" and d["split"] == "dev"]
    pools = {f"no-{x}": [d["key"] for d in dev if d["text"] != x] for x in meta["types"]}
    pools["all"] = [d["key"] for d in dev]
    jobs = [(S.out, p, ks, cfg["dagger_rounds"], cfg["train_budgets"], cfg["seed"]) for p, ks in pools.items()
            if ks and not os.path.exists(S.p("students", f"{p}.pkl"))]
    if not jobs:
        log("train: all students already trained - skipping"); return
    pmap(_train_job, jobs, "train: Imitation + DAgger (one unit = one leakage-free training pool)")


def _pools(meta):
    dev = [d for d in meta["instances"] if d["kind"] == "pure" and d["split"] == "dev"]
    pools = {f"no-{x}": [d["key"] for d in dev if d["text"] != x] for x in meta["types"]}
    pools["all"] = [d["key"] for d in dev]
    return dev, pools


def stage_tune():
    """FAIR TUNING. Every tunable policy tries its whole grid (at most TUNE_BUDGET settings, default first) on the
    same DEV sequences and train budgets the learned policies are trained on. The best mean ratio is chosen per
    pool ('no-X' = DEV of every type except X, used when testing on X). TEST sequences are never touched."""
    S, cfg = STORE, CFG; meta = S.meta(); R = [e for e in registry(cfg["ablations"]) if e.grid]
    dev, pools = _pools(meta)
    path = S.p("tuned", "grid.csv"); GID = {e.name: grid_id(e) for e in R}; old = None
    if os.path.exists(path):
        old = pd.read_csv(path, dtype={"grid_id": str})
        old = old[[GID.get(p) == g for p, g in zip(old.policy, old.grid_id)]]
    miss = [e.name for e in R if old is None or e.name not in set(old.policy)]
    if miss:
        n_set = sum(len(e.grid) for e in R if e.name in miss)
        log(f"tune: {len(miss)} policies, {n_set} settings, on {len(dev)} DEV sequences x budgets {cfg['train_budgets']}")
        rows = sum(pmap(_tune_job, [(S.out, d, cfg["train_budgets"], cfg["ablations"], miss) for d in dev],
                        f"tune (one unit = one DEV sequence x {len(cfg['train_budgets'])} budgets x {n_set} settings)"), [])
        new = pd.DataFrame(rows); new["grid_id"] = new.policy.map(GID)
        old = pd.concat([old, new], ignore_index=True) if old is not None else new
        old.to_csv(path, index=False)
    else:
        log("tune: every grid already evaluated - re-using it")
    tuned = choose_params(old, pools, R)
    atomic_write(S.p("tuned", "params.json"), json.dumps(tuned, indent=1, sort_keys=True))
    summ, eff = tuning_tables(old, R, tuned)
    summ.to_csv(S.p("results", "tuning_summary.csv"), index=False); eff.to_csv(S.p("results", "tuning_effects.csv"), index=False)
    for _, r in summ.iterrows():
        log(f"tune: {r.policy:16s} default {r.dev_default:.4f} -> chosen {r.dev_chosen:.4f}  {r.chosen}  (pools agree: {r.same_choice_all_pools})")
    for _, r in eff[eff.verdict.str.startswith("NO EFFECT")].iterrows():
        log(f"tune: WARNING - {r.policy}: setting '{r.setting}' never changes any DEV result (that part of the policy does nothing here)")


def _fid():
    return dict(STORE.fids(CFG["ablations"]), **{"Full cache": "-"})


def _prune_stale(pattern):
    """Before resuming a GPU test: drop saved rows of removed policies or rows computed with other settings."""
    FID = _fid(); n = 0
    for f in glob.glob(STORE.p("results", pattern)):
        if os.path.getsize(f) <= 1:
            continue
        d = pd.read_csv(f, dtype={"cfg_id": str})
        if "cfg_id" not in d:
            d["cfg_id"] = "-"
        keep = np.array([FID.get(p) == x for p, x in zip(d.policy, d.cfg_id.astype(str))], dtype=bool)
        if not keep.all() or "cfg_id" not in pd.read_csv(f, nrows=0).columns:
            n += int((~keep).sum()); d[keep].to_csv(f, index=False)
    if n:
        log(f"{pattern}: dropped {n} saved rows made with old settings or by a removed policy; they will be recomputed")


def stage_cpu():
    """T1-T4 + diagnostics. If a result file already exists, rows of removed policies and rows made with other
    settings are dropped, and only the missing policies are computed and appended."""
    S, cfg = STORE, CFG; abl = cfg["ablations"]; R = registry(abl); res = S.p("results"); insts = S.meta()["instances"]
    FID = _fid()

    def plan(f, names):
        path = os.path.join(res, f)
        if not os.path.exists(path):
            return names, None
        old = current_rows(pd.read_csv(path, dtype={"cfg_id": str}), FID)
        old = old.assign(cfg_id=old.cfg_id if "cfg_id" in old else "-")
        miss = [n for n in names if n not in set(old.policy)]
        log(f"cpu: {f} " + (f"exists - computing only new or re-tuned policies {miss}" if miss else "already complete - skipping (delete it to recompute)"))
        return miss, old

    def save(f, rows, old):
        df = pd.DataFrame(rows)
        pd.concat([old, df], ignore_index=True).to_csv(os.path.join(res, f), index=False) if old is not None else df.to_csv(os.path.join(res, f), index=False)

    names = [e.name for e in R]
    miss, old = plan("t1_ratio.csv", names)
    if miss:
        rows = sum(pmap(_ratio_job, [(S.out, d, cfg["test_budgets"], abl, miss) for d in insts],
                        f"T1 competitive ratio (one unit = one sequence x {len(cfg['test_budgets'])} budgets x {len(miss)} policies)"), [])
        save("t1_ratio.csv", rows, old)
    miss, old = plan("t2_stress.csv", names)
    if miss:
        rows = sum(pmap(_stress_job, [(S.out, cfg["stress_T"], s, cfg["stress_budgets"], abl, miss) for s in cfg["stress_seeds"]],
                        f"T2 stress traces (one unit = one seed x 6 traces x {len(miss)} policies)"), [])
        save("t2_stress.csv", rows, old)
    miss, old = plan("t3_adversary.csv", [n for n in names if n != "Optimal (offline)"])
    if miss:
        rows = pmap(_adv_job, [(S.out, n, cfg, abl) for n in miss], "T3 adversary (one unit = one policy)")
        save("t3_adversary.csv", rows, old)
    miss, old = plan("t4_overhead.csv", names)
    if miss:
        T = cfg["T"]; A = synthetic_attention(T, cfg["seed"]); k = int(0.2 * T); end = get_optimum(A, k); st = S.students("all"); rows = []
        tu = S.tuned("all")
        prog = Progress("T4 overhead (sequential, one process)", len(miss))
        for e in R:
            if e.name not in miss:
                continue
            tracemalloc.start(); _, dt, _ = simulate(A, k, e.make(Ctx(A=A, k=k, end=end, students=st, tuned=tu)))
            _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
            rows.append(dict(policy=e.name, family=e.family, us_per_step=dt * 1e6, peak_KB=peak / 1024, cfg_id=FID[e.name])); prog.tick(note=e.name)
        prog.finish(); save("t4_overhead.csv", rows, old)
    rp = os.path.join(res, "diag_chameleon_regret.csv"); redo = True
    if os.path.exists(rp) and os.path.getsize(rp) > 1:
        d = pd.read_csv(rp, dtype={"cfg_id": str}); redo = "cfg_id" not in d or (len(d) > 0 and str(d.cfg_id.iloc[0]) != FID["Chameleon"])
    if redo:
        out = pmap(_diag_job, [(S.out, d, cfg["test_budgets"][:3]) for d in insts if d["split"] == "test"],
                   "diagnostics: Chameleon regret, DAgger rounds")
        pd.DataFrame(sum([o[1] for o in out], [])).to_csv(os.path.join(res, "diag_chameleon_weights.csv"), index=False)
        pd.DataFrame(sum([o[2] for o in out], [])).to_csv(os.path.join(res, "diag_dagger_rounds.csv"), index=False)
        pd.DataFrame(sum([o[0] for o in out], [])).to_csv(os.path.join(res, "diag_chameleon_regret.csv"), index=False)


def _done_all(pattern, cols):
    return set().union(*[_done_set(f, cols) for f in glob.glob(STORE.p("results", pattern))])


def stage_ppl():
    """T5: real perplexity with a physically pruned cache, every policy, TEST sequences."""
    S, cfg = STORE, CFG; meta = S.meta(); R = registry(cfg["ablations"])
    if meta.get("synthetic"):
        log("T5 perplexity: skipped (synthetic data has no model)"); return
    by_text = {}
    for d in meta["instances"]:
        if d["split"] == "test":
            by_text.setdefault(d["text"], []).append(d)
    tasks = []
    for L in by_text.values():
        n = cfg["ppl_seqs"] if L[0]["kind"] == "pure" else cfg["ppl_shift_seqs"]
        tasks += [(d, b) for d in L[:n] for b in cfg["ppl_budgets"]]
    _prune_stale("t5_ppl_shard*.csv"); done = _done_all("t5_ppl_shard*.csv", ["key", "budget", "policy"])
    left = lambda d, b: [e for e in R if (d["key"], str(b), e.name) not in done]
    tasks = [(d, b) for d, b in tasks if left(d, b)]
    units = sum(len(left(d, b)) for d, b in tasks); FID = _fid()
    if not units:
        log("T5 perplexity: everything already saved - skipping"); return

    def work(i, n, report):
        tok, model = load_model(cfg); path = S.p("results", f"t5_ppl_shard{i}.csv")
        for d, b in tasks[i::n]:
            A, nll_full, ids = S.load_attn(d["key"]); T = len(ids); k = int(b * T)
            end, _ = S.load_opt(d["key"], b); st = S.students(pool_of(d)); tu = S.tuned(pool_of(d))
            ppl_full = float(np.exp(nll_full[k:].mean()))
            for e in left(d, b):
                t0 = time.time(); nll, _ = run_with_eviction(model, ids, k, e.make(Ctx(A=A, k=k, end=end, students=st, tuned=tu)))
                ppl = float(np.exp(nll[k:].mean()))
                _append_rows(path, [dict(key=d["key"], text=d["text"], kind=d["kind"], seq=d["seq"], budget=b, policy=e.name,
                                         family=e.family, ppl=ppl, ppl_full=ppl_full, dppl_pct=100 * (ppl / ppl_full - 1),
                                         cfg_id=FID[e.name])])
                report("tick", f"{d['key']} b={b} {e.name} ({time.time() - t0:.0f}s)")
    gpu_parallel("T5 perplexity (one unit = one policy decoding one sequence)", work, units)


def stage_needle():
    """T6: needle in a haystack, every policy, several budgets x depths x repetitions."""
    S, cfg = STORE, CFG; meta = S.meta(); R = registry(cfg["ablations"]); T = cfg["T"]
    if meta.get("synthetic"):
        log("T6 needle: skipped (synthetic data has no model)"); return
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(cfg["model"])
    filler = np.load(S.p("data/needle_filler.npy")).tolist()
    q_ids = tok.encode("\nQuestion: What is the secret code? Answer: The secret code is", add_special_tokens=False)
    rng = np.random.default_rng(cfg["seed"] + 7); needles = []
    for rep in range(cfg["needle_reps"]):
        for depth in cfg["needle_depths"]:
            code = str(rng.integers(1000, 9999)); nid = len(needles)
            needle = tok.encode(f" The secret code is {code}. ", add_special_tokens=False)
            n_fill = T - len(needle) - len(q_ids); pos = int(depth * n_fill)
            off = (nid * n_fill) % max(1, len(filler) - n_fill); f = filler[off:off + n_fill]
            ids = f[:pos] + needle + f[pos:] + q_ids; assert len(ids) == T, "needle filler too short"
            needles.append((nid, rep, depth, code, ids))
    _prune_stale("t6_needle_shard*.csv"); done = _done_all("t6_needle_shard*.csv", ["nid", "budget", "policy"]); FID = _fid()
    jobs = [("Full cache", 1.0)] + [(e.name, b) for b in cfg["needle_budgets"] for e in R]
    left = lambda nid: [(p, b) for p, b in jobs if (str(nid), str(b), p) not in done]
    todo = [x for x in needles if left(x[0])]
    units = sum(len(left(x[0])) for x in todo)
    if not units:
        log("T6 needle: everything already saved - skipping"); return
    st = S.students(f"no-{meta['filler_type']}")                 # learned policies never trained on the filler's text type
    tu = S.tuned(f"no-{meta['filler_type']}")                    # ... and tuned policies never tuned on it
    entries = {e.name: e for e in R}

    def work(i, n, report):
        tok, model = load_model(cfg); path = S.p("results", f"t6_needle_shard{i}.csv")
        for nid, rep, depth, code, ids in todo[i::n]:
            A, _ = attention_matrix(model, ids); ends = {}
            for pname, b in left(nid):
                if pname == "Full cache":
                    k, pol, fam = T, FullCache(), "reference"
                else:
                    k = int(b * T); ends.setdefault(b, get_optimum(A, k)); fam = entries[pname].family
                    pol = entries[pname].make(Ctx(A=A, k=k, end=ends[b], students=st, tuned=tu))
                _, gen = run_with_eviction(model, ids, k, pol, gen_tokens=6)
                ok = code in tok.decode(gen).replace(" ", "")
                _append_rows(path, [dict(nid=nid, rep=rep, depth=depth, budget=b, policy=pname, family=fam, success=ok, cfg_id=FID[pname])])
                report("tick", f"needle {nid} depth {depth} b={b} {pname}: {'found' if ok else 'missed'}")
    gpu_parallel("T6 needle (one unit = one policy on one needle at one budget)", work, units)

## 10. Start the run
This creates `OUT` and `config.json`. If `OUT` already has a config, that saved config is used (resume) and any option changes are ignored. Use a new `OUT` to change options.

In [ ]:
os.makedirs(OUT, exist_ok=True)
STORE = Store(OUT); LOG_FILE[0] = STORE.p("run_log.txt")
_cfg_path = STORE.p("config.json")
if os.path.exists(_cfg_path):
    CFG = json.load(open(_cfg_path))
    log(f"resuming run in {OUT} (preset '{CFG['preset']}', model {CFG['model']}, T={CFG['T']}); option changes are ignored")
else:
    CFG = dict(PRESETS[PRESET], preset=PRESET, ablations=bool(ABLATIONS), synthetic=bool(SYNTHETIC))
    if MODEL: CFG["model"] = MODEL
    if DTYPE: CFG["dtype"] = DTYPE
    CFG.update(OVERRIDES)
    assert set(CFG["ppl_budgets"]) <= set(CFG["test_budgets"]), "ppl_budgets must be a subset of test_budgets"
    assert set(CFG["train_budgets"]) <= set(CFG["test_budgets"]), "train_budgets must be a subset of test_budgets"
    atomic_write(_cfg_path, json.dumps(CFG, indent=1))
    log(f"new run in {OUT}: preset '{PRESET}', model {CFG['model']}, T={CFG['T']}, text types {CFG['text_types']}")
RUN_T0 = time.time()
print(f"progress is also written to {LOG_FILE[0]}  (tail -f it from a terminal)")

## 11. Data + attention extraction (GPU)
If a dataset fails to download, that text type is skipped with a message and the rest continue.

In [ ]:
stage_data()

## 12. Exact offline optimum for every (sequence, budget) (CPU, parallel)

In [ ]:
stage_optimum()

## 12b. Fair tuning on DEV sequences (CPU, parallel)
Every policy with settings tries at most 8 of them on the same DEV data; the best is chosen per leave-one-type-out pool. The log prints default vs chosen DEV ratio for every policy, and a WARNING for any setting that never changes a result. Tables: `results/tuning_summary.csv`, `results/tuning_effects.csv`; chosen settings: `tuned/params.json`.

In [ ]:
stage_tune()

## 13. Train Imitation and DAgger (CPU, parallel)

In [ ]:
stage_train()

## 14. CPU tests: T1 ratio, T2 stress, T3 adversary, T4 overhead, diagnostics

In [ ]:
stage_cpu()

## 15. T5 perplexity with a physically pruned cache (GPU, the slowest step)

In [ ]:
stage_ppl()

## 16. T6 needle in a haystack (GPU)

In [ ]:
stage_needle()

## 17. Report

In [ ]:
stage_report(STORE, CFG)
log(f"all stages done; total time this session {fmt_t(time.time() - RUN_T0)}")
display(Markdown(open(STORE.p("results", "summary.md")).read()))
for f in sorted(glob.glob(STORE.p("results", "*.png"))):
    print(os.path.basename(f)); display(Image(f))

## 18. Pack the results for download
The zip holds `results/` (CSVs, `summary.md`, plots), `config.json` and `run_log.txt`. The large cached attention matrices are left out.

In [ ]:
import shutil
tmp = STORE.p("_pack"); shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
shutil.copytree(STORE.p("results"), os.path.join(tmp, "results"))
for f in ["config.json", "run_log.txt"]:
    if os.path.exists(STORE.p(f)): shutil.copy(STORE.p(f), tmp)
print("saved", shutil.make_archive(OUT.rstrip("/") + "_results", "zip", tmp)); shutil.rmtree(tmp)